In [1]:
from pathlib import Path
import importlib.metadata
import json
import os
import platform
import sys
from IPython.display import display, Markdown
from rdkit import RDLogger

# Enumeration intentionally rejects many chemically invalid combinations.  RDKit
# reports every rejected intermediate through its logger; the validity counters
# below retain that information without flooding the persisted notebook output.
for channel in ("rdApp.debug", "rdApp.info", "rdApp.warning", "rdApp.error"):
    RDLogger.DisableLog(channel)

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "config" / "default.json").is_file():
    raise RuntimeError("Запускайте GOSHA.ipynb из директории TEST")

required = ["rdkit", "numpy", "pandas", "scikit-learn", "PyYAML"]
versions = {}
for package in required:
    try:
        versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError as exc:
        raise RuntimeError(
            f"Не установлен {package}. Выполните: ./venv/bin/pip install -r requirements.txt"
        ) from exc

print("project:", PROJECT_ROOT)
print("python:", sys.version.split()[0])
print("platform:", platform.platform())
print("packages:", versions)


project: /home/sigmatau17/TEST
python: 3.10.12
platform: Linux-5.15.167.4-microsoft-standard-WSL2-x86_64-with-glibc2.35
packages: {'rdkit': '2026.3.6', 'numpy': '2.2.6', 'pandas': '2.3.3', 'scikit-learn': '1.7.2', 'PyYAML': '6.0.3'}


In [2]:
FULL_RESULT_DIR = PROJECT_ROOT / "runs" / "full"
if (FULL_RESULT_DIR / "verification.json").is_file():
    full_verification = json.loads((FULL_RESULT_DIR / "verification.json").read_text())
    full_review = json.loads((FULL_RESULT_DIR / "review" / "review_summary.json").read_text())
    display({
        "acceptance_passed": full_verification["passed"],
        "generated_rows": full_verification["generated_rows"],
        "run_counts": full_verification["run_counts"],
        "zero_psoralen_cores": full_verification["checks"]["zero_psoralen_cores"],
        "independent_reviewed": full_review["reviewed"],
        "provisional": full_review["independent_joint_pass"],
        "selected": full_review["selected_after_physical_oracle"],
        "physical_oracle": full_review["physical_oracle"]["status"],
    })
else:
    print("Предварительный full-прогон не найден; notebook остаётся полностью исполнимым.")


{'acceptance_passed': True,
 'generated_rows': 9000,
 'run_counts': {'libinvent_rl:1701': 1000,
  'libinvent_rl:2903': 1000,
  'libinvent_rl:4211': 1000,
  'prior_random:1701': 1000,
  'prior_random:2903': 1000,
  'prior_random:4211': 1000,
  'weighted_retraining:1701': 1000,
  'weighted_retraining:2903': 1000,
  'weighted_retraining:4211': 1000},
 'zero_psoralen_cores': True,
 'independent_reviewed': 75,
 'provisional': 11,
 'selected': 0,
 'physical_oracle': 'not_run_external_tools_unavailable'}

In [3]:
import types

for loaded_name in list(sys.modules):
    if loaded_name == "mostgen" or loaded_name.startswith("mostgen."):
        del sys.modules[loaded_name]

embedded_package = types.ModuleType("mostgen")
embedded_package.__package__ = "mostgen"
embedded_package.__path__ = [str(PROJECT_ROOT / "mostgen")]
embedded_package.__file__ = str(PROJECT_ROOT / "mostgen" / "__init__.py")
sys.modules["mostgen"] = embedded_package

def _load_embedded_module(name: str, source: str, filename: Path):
    module = types.ModuleType(name)
    module.__file__ = str(filename)
    module.__package__ = name.rpartition(".")[0]
    sys.modules[name] = module
    exec(compile(source, str(filename), "exec"), module.__dict__)
    return module

print("Изолированный in-memory package namespace подготовлен")


Изолированный in-memory package namespace подготовлен


In [4]:
_init_source = r'''"""MOSTGen: reproducible UV/MOST computational screening prototype."""

__version__ = "0.1.0"

'''
exec(compile(_init_source, embedded_package.__file__, 'exec'), embedded_package.__dict__)
print('loaded mostgen', embedded_package.__version__)


loaded mostgen 0.1.0


In [5]:
_source = r'''from __future__ import annotations

import copy
import json
from pathlib import Path
from typing import Any


ROOT = Path(__file__).resolve().parents[1]
DEFAULT_CONFIG = ROOT / "config" / "default.json"


class ConfigError(ValueError):
    """Raised when an experiment configuration violates the contract."""


def _deep_update(base: dict[str, Any], patch: dict[str, Any]) -> dict[str, Any]:
    for key, value in patch.items():
        if isinstance(value, dict) and isinstance(base.get(key), dict):
            _deep_update(base[key], value)
        else:
            base[key] = value
    return base


def load_config(path: str | Path | None = None, mode: str = "full") -> dict[str, Any]:
    config_path = Path(path) if path else DEFAULT_CONFIG
    with config_path.open(encoding="utf-8") as handle:
        config = json.load(handle)
    config["_config_path"] = str(config_path.resolve())
    if mode == "smoke":
        smoke = copy.deepcopy(config["smoke"])
        config["execution"].update({k: v for k, v in smoke.items() if k != "training_rows_per_family"})
        config["execution"]["mode"] = "smoke"
        config["training_rows_per_family"] = smoke["training_rows_per_family"]
    elif mode == "full":
        config["execution"]["mode"] = "full"
        config["training_rows_per_family"] = 180
    elif mode == "production":
        config["execution"]["mode"] = "production"
        config["execution"]["backend"] = "reinvent4"
        config["training_rows_per_family"] = 180
    else:
        raise ConfigError(f"Unknown mode: {mode}")
    validate_config(config)
    return config


def apply_override(config: dict[str, Any], override_path: str | Path | None) -> dict[str, Any]:
    if not override_path:
        return config
    with Path(override_path).open(encoding="utf-8") as handle:
        patch = json.load(handle)
    merged = _deep_update(copy.deepcopy(config), patch)
    validate_config(merged)
    return merged


def validate_config(config: dict[str, Any]) -> None:
    required_families = {"nbd_qc", "dewar_pyrimidinone", "spiropyran"}
    missing = required_families - set(config.get("families", {}))
    if missing:
        raise ConfigError(f"Missing family configurations: {sorted(missing)}")
    project = config.get("project", {})
    if int(project.get("max_training_structures", 0)) > 30_000:
        raise ConfigError("max_training_structures exceeds the 30,000 structure contract")
    execution = config.get("execution", {})
    if float(execution.get("gpu_hours", 0.0)) > float(project.get("max_gpu_hours", 8.0)):
        raise ConfigError("Configured GPU budget exceeds project maximum")
    if int(execution.get("reviewer_budget_per_run", 0)) < int(execution.get("n_per_run", 0)):
        raise ConfigError("reviewer_budget_per_run must be at least n_per_run")
    if len(config.get("synthons", [])) < 2:
        raise ConfigError("At least two synthons are required")
    ids = [item["id"] for item in config["synthons"]]
    if len(ids) != len(set(ids)):
        raise ConfigError("Synthon identifiers must be unique")
    if not 0.0 < float(config["reward"]["floor"]) < 1.0:
        raise ConfigError("Reward floor must be between zero and one")


def dump_resolved_config(config: dict[str, Any], path: str | Path) -> None:
    clean = {k: v for k, v in config.items() if not k.startswith("_")}
    Path(path).write_text(json.dumps(clean, indent=2, sort_keys=True) + "\n", encoding="utf-8")

'''
_load_embedded_module('mostgen.config', _source, PROJECT_ROOT / 'mostgen' / 'config.py')
print('loaded mostgen.config')


loaded mostgen.config


In [6]:
_source = r'''from __future__ import annotations

import hashlib
import importlib.metadata
import json
import os
import platform
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

from . import __version__


def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def stable_hash(text: str, length: int = 16) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:length]


def package_versions(names: Iterable[str]) -> dict[str, str]:
    result: dict[str, str] = {}
    for name in names:
        try:
            result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            result[name] = "not-installed"
    return result


def git_state(root: Path) -> dict[str, Any]:
    try:
        commit = subprocess.run(
            ["git", "rev-parse", "HEAD"], cwd=root, check=True,
            text=True, capture_output=True, timeout=5,
        ).stdout.strip()
        dirty = bool(subprocess.run(
            ["git", "status", "--porcelain"], cwd=root, check=True,
            text=True, capture_output=True, timeout=5,
        ).stdout.strip())
        return {"commit": commit, "dirty": dirty}
    except (OSError, subprocess.SubprocessError):
        return {"commit": "not-a-git-worktree", "dirty": None}


def write_manifest(
    path: str | Path,
    config: dict[str, Any],
    inputs: Iterable[str | Path] = (),
    command: list[str] | None = None,
) -> dict[str, Any]:
    root = Path(__file__).resolve().parents[1]
    input_records = []
    for item in inputs:
        source = Path(item)
        if source.exists() and source.is_file():
            input_records.append({
                "path": str(source.resolve()),
                "bytes": source.stat().st_size,
                "sha256": sha256_file(source),
            })
    manifest = {
        "schema_version": "1.0",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "mostgen_version": __version__,
        "python": sys.version,
        "platform": platform.platform(),
        "executable": sys.executable,
        "command": command or sys.argv,
        "cwd": str(Path.cwd()),
        "timezone": os.environ.get("TZ", "system-default"),
        "random_seeds": list(config["execution"]["seeds"]),
        "budget": {
            "max_training_structures": config["project"]["max_training_structures"],
            "max_gpu_hours": config["project"]["max_gpu_hours"],
            "configured_gpu_hours": config["execution"]["gpu_hours"],
            "reviewer_calls_per_run": config["execution"]["reviewer_budget_per_run"],
        },
        "backend": config["execution"]["backend"],
        "packages": package_versions(["rdkit", "numpy", "pandas", "scikit-learn", "PyYAML"]),
        "git": git_state(root),
        "inputs": input_records,
        "sources": [
            {
                **source,
                "version": source.get("version", "REFERENCE_UNPINNED"),
                "checksum": source.get("checksum", "NOT_FETCHED_NO_LOCAL_ARTIFACT"),
            }
            for source in config["sources"]
        ],
    }
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return manifest


def write_artifact_manifest(root: str | Path) -> dict[str, Any]:
    experiment = Path(root).resolve()
    destination = experiment / "artifact_manifest.json"
    artifacts = []
    for path in sorted(experiment.rglob("*")):
        if not path.is_file() or path == destination:
            continue
        artifacts.append({
            "path": str(path.relative_to(experiment)),
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        })
    manifest = {"schema_version": "1.0", "root": str(experiment), "artifacts": artifacts}
    destination.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return manifest


def append_event(path: str | Path, event: str, payload: dict[str, Any]) -> None:
    record = {
        "time_utc": datetime.now(timezone.utc).isoformat(),
        "event": event,
        **payload,
    }
    with Path(path).open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, sort_keys=True) + "\n")
'''
_load_embedded_module('mostgen.provenance', _source, PROJECT_ROOT / 'mostgen' / 'provenance.py')
print('loaded mostgen.provenance')


loaded mostgen.provenance


In [7]:
_source = r'''from __future__ import annotations

from dataclasses import asdict, dataclass
from functools import lru_cache
from typing import Any, Iterable

from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, Lipinski, rdMolDescriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator


class ChemistryError(ValueError):
    """A molecule cannot be standardized or does not satisfy the chemistry contract."""


@dataclass(frozen=True)
class MoleculePair:
    smiles: str
    charged_smiles: str
    family: str
    synthon_a: str
    synthon_b: str
    formula: str


@dataclass(frozen=True)
class SafetyVeto:
    veto: bool
    psoralen_alert: bool
    known_phototoxic_match: bool
    psoralen_similarity: float
    psoralen_similarity_warning: bool
    reactive_alerts: tuple[str, ...]
    unsupported_elements: tuple[str, ...]
    reasons: tuple[str, ...]

    def to_dict(self) -> dict[str, Any]:
        result = asdict(self)
        result["reactive_alerts"] = ";".join(self.reactive_alerts)
        result["unsupported_elements"] = ";".join(self.unsupported_elements)
        result["reasons"] = ";".join(self.reasons)
        return result


def parse_smiles(smiles: str) -> Chem.Mol:
    if not isinstance(smiles, str) or not smiles.strip():
        raise ChemistryError("empty_smiles")
    mol = Chem.MolFromSmiles(smiles.strip(), sanitize=True)
    if mol is None:
        raise ChemistryError("invalid_smiles")
    return mol


@lru_cache(maxsize=100_000)
def standardize_smiles(smiles: str) -> str:
    mol = parse_smiles(smiles)
    try:
        mol = rdMolStandardize.Cleanup(mol)
        mol = rdMolStandardize.FragmentParent(mol)
        uncharger = rdMolStandardize.Uncharger(canonicalOrder=True)
        # Preserve genuine zwitterions: only accept uncharging when net formal
        # charge is non-zero.  Merocyanines contain balanced explicit charges.
        if Chem.GetFormalCharge(mol) != 0:
            mol = uncharger.uncharge(mol)
        Chem.SanitizeMol(mol)
    except Exception as exc:  # RDKit exposes several sanitizer exception types.
        raise ChemistryError(f"standardization_failed:{exc}") from exc
    return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)


def molecular_formula(smiles: str) -> str:
    return rdMolDescriptors.CalcMolFormula(parse_smiles(smiles))


def build_pair(config: dict[str, Any], family: str, synthon_a: str, synthon_b: str) -> MoleculePair:
    if family not in config["families"]:
        raise ChemistryError(f"unsupported_family:{family}")
    library = {entry["id"]: entry for entry in config["synthons"]}
    try:
        first, second = library[synthon_a], library[synthon_b]
    except KeyError as exc:
        raise ChemistryError(f"unknown_synthon:{exc.args[0]}") from exc
    if not first.get("commercial") or not second.get("commercial"):
        raise ChemistryError("noncommercial_synthon")
    family_config = config["families"][family]
    raw_ground = family_config["ground_template"].format(r1=first["smiles"], r2=second["smiles"])
    raw_charged = family_config["charged_template"].format(r1=first["smiles"], r2=second["smiles"])
    ground = standardize_smiles(raw_ground)
    charged = standardize_smiles(raw_charged)
    ground_formula = molecular_formula(ground)
    charged_formula = molecular_formula(charged)
    if ground_formula != charged_formula:
        raise ChemistryError(f"isomer_formula_mismatch:{ground_formula}!={charged_formula}")
    if ground == charged:
        raise ChemistryError("isomer_pair_identical")
    return MoleculePair(ground, charged, family, synthon_a, synthon_b, ground_formula)


def murcko_scaffold(smiles: str) -> str:
    mol = parse_smiles(smiles)
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    if scaffold.GetNumAtoms() == 0:
        return Chem.MolToSmiles(mol, canonical=True)
    generic = MurckoScaffold.MakeScaffoldGeneric(scaffold)
    return Chem.MolToSmiles(generic, canonical=True)


@lru_cache(maxsize=8)
def _morgan_generator(bits: int):
    return GetMorganGenerator(radius=2, fpSize=bits, includeChirality=True)


@lru_cache(maxsize=100_000)
def fingerprint(smiles: str, bits: int = 256):
    return _morgan_generator(bits).GetFingerprint(parse_smiles(smiles))


def fingerprint_indices(smiles: str, bits: int = 256) -> list[int]:
    return list(fingerprint(smiles, bits).GetOnBits())


def tanimoto(smiles_a: str, smiles_b: str, bits: int = 256) -> float:
    return float(DataStructs.TanimotoSimilarity(fingerprint(smiles_a, bits), fingerprint(smiles_b, bits)))


def max_similarity(smiles: str, references: Iterable[str], bits: int = 256) -> float:
    query = fingerprint(smiles, bits)
    ref_fps = [fingerprint(reference, bits) for reference in references]
    if not ref_fps:
        return 0.0
    return float(max(DataStructs.BulkTanimotoSimilarity(query, ref_fps)))


def descriptors(smiles: str) -> dict[str, float]:
    mol = parse_smiles(smiles)
    atoms = max(1, mol.GetNumHeavyAtoms())
    rings = float(rdMolDescriptors.CalcNumRings(mol))
    aromatic = float(rdMolDescriptors.CalcNumAromaticRings(mol))
    rotors = float(Lipinski.NumRotatableBonds(mol))
    hetero = float(rdMolDescriptors.CalcNumHeteroatoms(mol))
    charge_separation = float(sum(abs(atom.GetFormalCharge()) for atom in mol.GetAtoms()))
    return {
        "mol_wt": float(Descriptors.MolWt(mol)),
        "logp": float(Descriptors.MolLogP(mol)),
        "tpsa": float(rdMolDescriptors.CalcTPSA(mol)),
        "hbd": float(Lipinski.NumHDonors(mol)),
        "hba": float(Lipinski.NumHAcceptors(mol)),
        "rings": rings,
        "aromatic_rings": aromatic,
        "rotatable_bonds": rotors,
        "hetero_fraction": hetero / atoms,
        "fraction_csp3": float(rdMolDescriptors.CalcFractionCSP3(mol)),
        "formal_charge": float(Chem.GetFormalCharge(mol)),
        "charge_separation": charge_separation,
        "heavy_atoms": float(atoms),
    }


def synthetic_accessibility(smiles: str) -> float:
    """Transparent Ertl-inspired proxy on a 1 (easy) to 10 (hard) scale.

    This is deliberately named and reported as a proxy; production review may
    replace it with the RDKit contrib SA implementation without changing the
    CSV contract.
    """
    d = descriptors(smiles)
    raw = (
        1.0
        + 0.20 * d["rings"]
        + 0.13 * d["rotatable_bonds"]
        + 0.035 * max(0.0, d["heavy_atoms"] - 12.0)
        + 0.18 * abs(d["logp"] - 2.0)
        + 0.12 * d["charge_separation"]
    )
    return min(10.0, max(1.0, raw))


@lru_cache(maxsize=256)
def _query(pattern: str) -> Chem.Mol | None:
    query = Chem.MolFromSmarts(pattern)
    return query if query is not None else Chem.MolFromSmiles(pattern)


@lru_cache(maxsize=64)
def _core_query(pattern: str) -> Chem.Mol | None:
    """A sanitized aromatic topology query for fused-core matching."""
    return Chem.MolFromSmiles(pattern)


def safety_veto(smiles: str, config: dict[str, Any]) -> SafetyVeto:
    reasons: list[str] = []
    try:
        mol = parse_smiles(smiles)
    except ChemistryError as exc:
        return SafetyVeto(True, False, False, 0.0, False, (), (), (str(exc),))

    allowed = set(config["elements"])
    unsupported = tuple(sorted({atom.GetSymbol() for atom in mol.GetAtoms()} - allowed))
    if unsupported:
        reasons.append("unsupported_elements")

    safety = config["safety"]
    psoralen = False
    for pattern in safety["psoralen_core_smarts"]:
        query = _core_query(pattern)
        if query is not None and mol.HasSubstructMatch(query):
            psoralen = True
            reasons.append("psoralen_or_furocoumarin_core")
            break

    canonical = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
    known = False
    known_canonical: list[str] = []
    for known_smiles in safety["known_phototoxic_smiles"]:
        try:
            normalized_known = standardize_smiles(known_smiles)
            known_canonical.append(normalized_known)
            if canonical == normalized_known:
                known = True
                reasons.append("known_phototoxic_structure")
        except ChemistryError:
            continue
    psoralen_similarity = max_similarity(canonical, known_canonical) if known_canonical else 0.0
    psoralen_similarity_warning = (
        not psoralen
        and not known
        and psoralen_similarity >= float(safety.get("psoralen_similarity_warning_threshold", 0.45))
    )

    reactive: list[str] = []
    for pattern in safety["reactive_fragments"]:
        query = _query(pattern)
        if query is not None and mol.HasSubstructMatch(query):
            reactive.append(pattern)
    if reactive:
        reasons.append("reactive_or_unstable_alert")

    return SafetyVeto(
        veto=bool(psoralen or known or reactive or unsupported),
        psoralen_alert=psoralen,
        known_phototoxic_match=known,
        psoralen_similarity=psoralen_similarity,
        psoralen_similarity_warning=psoralen_similarity_warning,
        reactive_alerts=tuple(reactive),
        unsupported_elements=unsupported,
        reasons=tuple(reasons),
    )


def validate_pair(pair: MoleculePair, config: dict[str, Any]) -> tuple[bool, list[str]]:
    reasons: list[str] = []
    try:
        ground_formula = molecular_formula(pair.smiles)
        charged_formula = molecular_formula(pair.charged_smiles)
        if ground_formula != charged_formula:
            reasons.append("isomer_formula_mismatch")
        if pair.smiles == pair.charged_smiles:
            reasons.append("isomer_pair_identical")
    except ChemistryError as exc:
        reasons.append(str(exc))
    veto = safety_veto(pair.smiles, config)
    reasons.extend(veto.reasons)
    return not reasons, sorted(set(reasons))
'''
_load_embedded_module('mostgen.chemistry', _source, PROJECT_ROOT / 'mostgen' / 'chemistry.py')
print('loaded mostgen.chemistry')


loaded mostgen.chemistry


In [8]:
_source = r'''from __future__ import annotations

import math
from statistics import fmean
from typing import Iterable, Sequence


def trapezoid_auc(wavelengths: Sequence[float], absorbance: Sequence[float], low: float, high: float) -> float:
    if len(wavelengths) != len(absorbance) or len(wavelengths) < 2:
        raise ValueError("wavelength and absorbance arrays must have equal length >= 2")
    if any(b <= a for a, b in zip(wavelengths, wavelengths[1:])):
        raise ValueError("wavelengths must be strictly increasing")
    if low >= high:
        raise ValueError("integration interval must have positive width")
    points: list[tuple[float, float]] = []
    for x, y in zip(wavelengths, absorbance):
        if low <= x <= high:
            points.append((float(x), float(y)))
    for edge in (low, high):
        if not any(abs(x - edge) < 1e-12 for x, _ in points):
            for index in range(len(wavelengths) - 1):
                left, right = wavelengths[index], wavelengths[index + 1]
                if left <= edge <= right:
                    ratio = (edge - left) / (right - left)
                    y = absorbance[index] + ratio * (absorbance[index + 1] - absorbance[index])
                    points.append((edge, float(y)))
                    break
    points.sort()
    return sum((x2 - x1) * (y1 + y2) / 2.0 for (x1, y1), (x2, y2) in zip(points, points[1:]))


def critical_wavelength(wavelengths: Sequence[float], absorbance: Sequence[float], fraction: float = 0.90) -> float:
    if not 0.0 < fraction < 1.0:
        raise ValueError("fraction must be between zero and one")
    total = trapezoid_auc(wavelengths, absorbance, wavelengths[0], wavelengths[-1])
    if total <= 0.0:
        return float(wavelengths[0])
    target = total * fraction
    cumulative = 0.0
    for (x1, y1), (x2, y2) in zip(zip(wavelengths, absorbance), zip(wavelengths[1:], absorbance[1:])):
        segment = (x2 - x1) * (y1 + y2) / 2.0
        if cumulative + segment >= target:
            if segment <= 0.0:
                return float(x2)
            return float(x1 + (x2 - x1) * (target - cumulative) / segment)
        cumulative += segment
    return float(wavelengths[-1])


def beer_lambert_transmittance(absorbance: Iterable[float], loading_scale: float = 1.0) -> float:
    values = [10.0 ** (-max(0.0, float(value)) * loading_scale) for value in absorbance]
    return fmean(values) if values else 1.0


def sigmoid(value: float, midpoint: float, scale: float) -> float:
    if scale <= 0:
        raise ValueError("scale must be positive")
    exponent = max(-60.0, min(60.0, -(value - midpoint) / scale))
    return 1.0 / (1.0 + math.exp(exponent))


def interval_score(value: float, low: float, high: float, softness: float) -> float:
    if not low < high:
        raise ValueError("low must be less than high")
    return sigmoid(value, low, softness) * sigmoid(high - value, 0.0, softness)


def weighted_geometric_mean(
    components: dict[str, float],
    weights: dict[str, float],
    floor: float = 1e-3,
) -> float:
    if not components:
        return 0.0
    numerator = 0.0
    denominator = 0.0
    for key, raw in components.items():
        weight = float(weights.get(key, 1.0))
        if weight <= 0.0:
            continue
        value = max(floor, min(1.0, float(raw)))
        numerator += weight * math.log(value)
        denominator += weight
    return math.exp(numerator / denominator) if denominator else 0.0


def lower_confidence_bound(mean: float, std: float, z: float = 1.645) -> float:
    return float(mean) - float(z) * max(0.0, float(std))


def upper_confidence_bound(mean: float, std: float, z: float = 1.645) -> float:
    return float(mean) + float(z) * max(0.0, float(std))

'''
_load_embedded_module('mostgen.numerics', _source, PROJECT_ROOT / 'mostgen' / 'numerics.py')
print('loaded mostgen.numerics')


loaded mostgen.numerics


In [9]:
_source = r'''from __future__ import annotations

import csv
import json
import math
import random
from pathlib import Path
from typing import Any, Iterable

from .chemistry import build_pair, descriptors, murcko_scaffold, safety_veto
from .numerics import critical_wavelength, trapezoid_auc
from .provenance import sha256_file, stable_hash


WAVELENGTHS = tuple(range(290, 401, 5))


def write_csv(path: str | Path, rows: Iterable[dict[str, Any]], fieldnames: list[str] | None = None) -> int:
    materialized = list(rows)
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if fieldnames is None:
        fieldnames = list(materialized[0]) if materialized else []
    with destination.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames, extrasaction="ignore")
        if fieldnames:
            writer.writeheader()
            writer.writerows(materialized)
    return len(materialized)


def read_csv(path: str | Path) -> list[dict[str, str]]:
    with Path(path).open(encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))


def _noise(key: str, magnitude: float) -> float:
    integer = int(stable_hash(key, 12), 16)
    return magnitude * (2.0 * (integer / float(16**12 - 1)) - 1.0)


def _gaussian(x: float, centre: float, width: float, amplitude: float) -> float:
    return amplitude * math.exp(-0.5 * ((x - centre) / width) ** 2)


def reference_labels(smiles: str, charged_smiles: str, family: str) -> dict[str, float]:
    """Deterministic fixture labels used only by the smoke backend.

    Values are smooth functions of RDKit descriptors plus deterministic noise,
    which makes leakage/error tests meaningful without pretending that the
    records are experimental observations.
    """
    d = descriptors(smiles)
    q = descriptors(charged_smiles)
    family_peak = {"nbd_qc": (314.0, 366.0), "dewar_pyrimidinone": (307.0, 354.0), "spiropyran": (325.0, 382.0)}[family]
    family_energy = {"nbd_qc": 91.0, "dewar_pyrimidinone": 55.0, "spiropyran": 32.0}[family]
    family_log_half = {"nbd_qc": 1.02, "dewar_pyrimidinone": 0.78, "spiropyran": 0.62}[family]
    shift = 3.5 * d["aromatic_rings"] + 0.065 * d["tpsa"] + 1.7 * d["logp"]
    donor_acceptor = min(2.0, d["hba"] / 3.0 + d["hbd"] / 2.0)
    peak_a = family_peak[0] + 0.30 * shift + _noise(smiles + "p1", 4.0)
    peak_b = family_peak[1] + 0.65 * shift + _noise(smiles + "p2", 6.0)
    amp_a = 0.45 + 0.065 * d["aromatic_rings"] + 0.025 * donor_acceptor
    amp_b = 0.22 + 0.095 * donor_acceptor + 0.035 * d["aromatic_rings"]
    spectrum = [
        max(0.0, _gaussian(w, peak_a, 17.0, amp_a) + _gaussian(w, peak_b, 29.0, amp_b) + _noise(f"{smiles}:{w}", 0.012))
        for w in WAVELENGTHS
    ]
    uvb_auc = trapezoid_auc(WAVELENGTHS, spectrum, 290.0, 320.0)
    uva_auc = trapezoid_auc(WAVELENGTHS, spectrum, 320.0, 400.0)
    lambda_c = critical_wavelength(WAVELENGTHS, spectrum)
    delta_descriptor = abs(q["fraction_csp3"] - d["fraction_csp3"]) + abs(q["charge_separation"] - d["charge_separation"]) * 0.25
    energy = max(3.0, family_energy + 7.0 * delta_descriptor + 1.2 * d["aromatic_rings"] - 0.035 * d["mol_wt"] + _noise(smiles + "dh", 8.0))
    specific = energy * 277.7777778 / max(1.0, d["mol_wt"])
    log_half = family_log_half + 0.10 * d["aromatic_rings"] + 0.04 * d["logp"] - 0.025 * d["rotatable_bonds"] + _noise(smiles + "t", 0.24)
    kp = -5.30 + 0.42 * d["logp"] - 0.012 * d["mol_wt"] - 0.018 * d["tpsa"] + _noise(smiles + "kp", 0.30)
    photo_logit = -2.2 + 0.58 * d["aromatic_rings"] + 0.20 * max(0.0, d["logp"] - 3.0) + 0.15 * donor_acceptor
    phototoxicity = 1.0 / (1.0 + math.exp(-photo_logit))
    phototoxicity = min(0.98, max(0.02, phototoxicity + _noise(smiles + "pt", 0.08)))
    result = {
        "uvb_auc": uvb_auc,
        "uva_auc": uva_auc,
        "lambda_c_nm": lambda_c,
        "energy_kj_mol": energy,
        "specific_energy_wh_kg": specific,
        "log_half_life_h": log_half,
        "kp_log_cm_s": kp,
        "phototoxicity_probability": phototoxicity,
    }
    result.update({f"abs_{w}": value for w, value in zip(WAVELENGTHS, spectrum)})
    return result


def scaffold_split(scaffold: str) -> str:
    bucket = int(stable_hash(scaffold, 8), 16) % 10
    if bucket < 7:
        return "train"
    if bucket < 9:
        return "validation"
    return "test"


def enumerate_library(config: dict[str, Any]) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    synthon_ids = [entry["id"] for entry in config["synthons"]]
    for family in sorted(config["families"]):
        for first in synthon_ids:
            for second in synthon_ids:
                try:
                    pair = build_pair(config, family, first, second)
                except ValueError:
                    continue
                veto = safety_veto(pair.smiles, config)
                if veto.veto:
                    continue
                rows.append({
                    "candidate_id": stable_hash(f"{family}|{first}|{second}", 20),
                    "family": family,
                    "synthon_a": first,
                    "synthon_b": second,
                    "smiles": pair.smiles,
                    "charged_smiles": pair.charged_smiles,
                    "formula": pair.formula,
                    "scaffold": murcko_scaffold(pair.smiles),
                })
    # Canonical structure identity wins over alternate synthon encodings.
    unique: dict[tuple[str, str], dict[str, Any]] = {}
    for row in rows:
        unique.setdefault((row["family"], row["smiles"]), row)
    return list(unique.values())


def _ensure_split_coverage(rows: list[dict[str, Any]]) -> None:
    """Keep scaffolds intact while ensuring every split is populated globally."""
    scaffolds = sorted({row["scaffold"] for row in rows})
    assignments = {scaffold: scaffold_split(scaffold) for scaffold in scaffolds}
    present = set(assignments.values())
    for wanted, index in (("validation", -2), ("test", -1)):
        if wanted not in present and len(scaffolds) >= abs(index):
            assignments[scaffolds[index]] = wanted
    for row in rows:
        row["split"] = assignments[row["scaffold"]]


def prepare_data(config: dict[str, Any], output_dir: str | Path, root: str | Path) -> dict[str, Any]:
    destination = Path(output_dir)
    destination.mkdir(parents=True, exist_ok=True)
    root_path = Path(root)
    library = enumerate_library(config)
    per_family = int(config["training_rows_per_family"])
    rng = random.Random(int(config["project"]["default_seed"]))
    selected: list[dict[str, Any]] = []
    for family in sorted(config["families"]):
        members = [row for row in library if row["family"] == family]
        rng.shuffle(members)
        selected.extend(members[:per_family])
    if len(selected) > int(config["project"]["max_training_structures"]):
        raise ValueError("Prepared training corpus exceeds configured 30,000-row cap")
    _ensure_split_coverage(selected)
    source_by_family = {"nbd_qc": "M01", "dewar_pyrimidinone": "M03/M04", "spiropyran": "M01/M04"}
    training_rows: list[dict[str, Any]] = []
    for row in selected:
        labels = reference_labels(row["smiles"], row["charged_smiles"], row["family"])
        training_rows.append({
            **row,
            "source_id": source_by_family[row["family"]],
            "source_kind": "deterministic_fixture",
            "evidence_tier": "synthetic_smoke_only",
            "state": "ground_and_charged_pair",
            "temperature_k": 305.0,
            "medium": "conditional_standard_medium",
            "label_level": "molecule_state_condition",
            **labels,
        })
    training_path = destination / "reviewer_training.csv"
    write_csv(training_path, training_rows)
    library_path = destination / "reaction_library.csv"
    write_csv(library_path, library)
    local_inputs = []
    for name in ("database_matrix_MOST_UV_skin.xlsx", "Задание.docx", "Солнцезащитная плёнка с молекулярным накоплением солнечной энергии.pptx"):
        path = root_path / name
        if path.exists():
            local_inputs.append({"path": str(path.resolve()), "bytes": path.stat().st_size, "sha256": sha256_file(path)})
    provenance = {
        "schema_version": "1.0",
        "raw_inputs_mutated": False,
        "derivative_data": str(training_path.resolve()),
        "training_rows": len(training_rows),
        "library_rows": len(library),
        "local_inputs": local_inputs,
        "source_registry": config["sources"],
        "warning": "Fixture labels are synthetic and cannot support scientific, efficacy, or safety claims.",
    }
    (destination / "data_manifest.json").write_text(json.dumps(provenance, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return provenance
'''
_load_embedded_module('mostgen.data', _source, PROJECT_ROOT / 'mostgen' / 'data.py')
print('loaded mostgen.data')


loaded mostgen.data


In [10]:
_source = r'''from __future__ import annotations

import json
import math
import pickle
from dataclasses import dataclass
from pathlib import Path
from statistics import median
from typing import Any, Iterable

import numpy as np
from rdkit import DataStructs
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

from .chemistry import descriptors, fingerprint, max_similarity
from .data import WAVELENGTHS, read_csv
from .provenance import stable_hash


DESCRIPTOR_NAMES = (
    "mol_wt", "logp", "tpsa", "hbd", "hba", "rings", "aromatic_rings",
    "rotatable_bonds", "hetero_fraction", "fraction_csp3", "formal_charge",
    "charge_separation", "heavy_atoms",
)
FAMILIES = ("nbd_qc", "dewar_pyrimidinone", "spiropyran")
SPECTRAL_TARGETS = tuple(f"abs_{w}" for w in WAVELENGTHS)
MOST_TARGETS = ("energy_kj_mol", "specific_energy_wh_kg", "log_half_life_h")
SAFETY_TARGETS = ("kp_log_cm_s", "phototoxicity_probability")


def feature_vector(smiles: str, family: str, bits: int) -> np.ndarray:
    fp = fingerprint(smiles, bits)
    fp_array = np.zeros(bits, dtype=np.float32)
    DataStructs.ConvertToNumpyArray(fp, fp_array)
    desc = descriptors(smiles)
    scaled = np.asarray([
        desc["mol_wt"] / 500.0,
        desc["logp"] / 6.0,
        desc["tpsa"] / 150.0,
        desc["hbd"] / 5.0,
        desc["hba"] / 10.0,
        desc["rings"] / 8.0,
        desc["aromatic_rings"] / 6.0,
        desc["rotatable_bonds"] / 12.0,
        desc["hetero_fraction"],
        desc["fraction_csp3"],
        desc["formal_charge"] / 3.0,
        desc["charge_separation"] / 4.0,
        desc["heavy_atoms"] / 60.0,
    ], dtype=np.float32)
    one_hot = np.asarray([1.0 if family == item else 0.0 for item in FAMILIES], dtype=np.float32)
    return np.concatenate([fp_array, scaled, one_hot])


@dataclass
class Prediction:
    mean: dict[str, float]
    std: dict[str, float]


class ReviewerBundle:
    """Serializable independent ensemble of spectrum, MOST, and safety models."""

    def __init__(
        self,
        purpose: str,
        bits: int,
        spectrum_models: list[Any],
        safety_models: list[Any],
        most_models: dict[str, list[Any]],
        references: dict[str, Any],
        metrics: dict[str, Any],
    ) -> None:
        self.purpose = purpose
        self.bits = bits
        self.spectrum_models = spectrum_models
        self.safety_models = safety_models
        self.most_models = most_models
        self.references = references
        self.metrics = metrics

    @staticmethod
    def _ensemble_predict(models: list[Any], vector: np.ndarray, targets: tuple[str, ...]) -> Prediction:
        member_values = []
        sample = vector.reshape(1, -1)
        for model in models:
            member_values.append(np.asarray(model.predict(sample)[0], dtype=float).reshape(-1))
        values = np.vstack(member_values)
        return Prediction(
            mean={key: float(value) for key, value in zip(targets, values.mean(axis=0))},
            std={key: float(value) for key, value in zip(targets, values.std(axis=0, ddof=0))},
        )

    def predict(self, smiles: str, family: str) -> dict[str, Any]:
        return self.predict_many([(smiles, family)])[0]

    def predict_many(self, molecules: list[tuple[str, str]]) -> list[dict[str, Any]]:
        if not molecules:
            return []
        x = np.vstack([feature_vector(smiles, family, self.bits) for smiles, family in molecules])

        def batch(models: list[Any], targets: tuple[str, ...], indices: list[int] | None = None):
            selected_x = x if indices is None else x[indices]
            values = np.asarray([model.predict(selected_x) for model in models], dtype=float)
            means, stds = values.mean(axis=0), values.std(axis=0, ddof=0)
            return [
                Prediction(
                    mean={key: float(value) for key, value in zip(targets, means[index].reshape(-1))},
                    std={key: float(value) for key, value in zip(targets, stds[index].reshape(-1))},
                )
                for index in range(len(selected_x))
            ]

        spectrum = batch(self.spectrum_models, SPECTRAL_TARGETS)
        safety = batch(self.safety_models, SAFETY_TARGETS)
        most: list[Prediction | None] = [None] * len(molecules)
        for family in FAMILIES:
            indices = [index for index, (_, item_family) in enumerate(molecules) if item_family == family]
            if not indices:
                continue
            predictions = batch(self.most_models[family], MOST_TARGETS, indices)
            for index, prediction in zip(indices, predictions):
                most[index] = prediction
        results = []
        spectral_refs = self.references["spectral_smiles"]
        for index, (smiles, family) in enumerate(molecules):
            results.append({
                "spectrum": spectrum[index], "most": most[index], "safety": safety[index],
                "ad_spectral_similarity": max_similarity(smiles, spectral_refs, self.bits),
                "ad_most_similarity": max_similarity(smiles, self.references["most_smiles_by_family"].get(family, []), self.bits),
                "family_reference_medians": self.references["family_medians"][family],
                "reviewer_purpose": self.purpose,
            })
        return results

    def save(self, path: str | Path) -> None:
        destination = Path(path)
        destination.parent.mkdir(parents=True, exist_ok=True)
        with destination.open("wb") as handle:
            pickle.dump(self, handle, protocol=pickle.HIGHEST_PROTOCOL)

    @classmethod
    def load(cls, path: str | Path) -> "ReviewerBundle":
        with Path(path).open("rb") as handle:
            bundle = pickle.load(handle)
        if not isinstance(bundle, cls):
            raise TypeError("Reviewer artifact has an unexpected type")
        return bundle


def _matrix(rows: list[dict[str, str]], bits: int) -> np.ndarray:
    return np.vstack([feature_vector(row["smiles"], row["family"], bits) for row in rows])


def _targets(rows: list[dict[str, str]], names: tuple[str, ...]) -> np.ndarray:
    return np.asarray([[float(row[name]) for name in names] for row in rows], dtype=np.float64)


def _new_model(kind: str, seed: int):
    common = dict(n_estimators=28, random_state=seed, n_jobs=1, min_samples_leaf=2, max_features=0.65)
    if kind == "reward":
        return RandomForestRegressor(bootstrap=True, **common)
    return ExtraTreesRegressor(bootstrap=False, **common)


def _fit_members(kind: str, count: int, seed: int, x: np.ndarray, y: np.ndarray) -> list[Any]:
    models = []
    for index in range(count):
        model = _new_model(kind, seed + 1009 * index)
        model.fit(x, y)
        models.append(model)
    return models


def _predict_mean(models: list[Any], x: np.ndarray) -> np.ndarray:
    return np.mean([np.asarray(model.predict(x), dtype=float) for model in models], axis=0)


def _split_metrics(
    rows: list[dict[str, str]],
    bits: int,
    spectrum_models: list[Any],
    safety_models: list[Any],
    most_models: dict[str, list[Any]],
) -> dict[str, Any]:
    results: dict[str, Any] = {}
    for split in ("validation", "test"):
        subset = [row for row in rows if row["split"] == split]
        if not subset:
            results[split] = {"n": 0, "status": "not_available"}
            continue
        x = _matrix(subset, bits)
        result: dict[str, Any] = {"n": len(subset)}
        for group, targets, models in (
            ("spectrum", SPECTRAL_TARGETS, spectrum_models),
            ("safety", SAFETY_TARGETS, safety_models),
        ):
            true = _targets(subset, targets)
            pred = _predict_mean(models, x)
            result[group] = {
                "mae": float(mean_absolute_error(true, pred)),
                "rmse": float(root_mean_squared_error(true, pred)),
            }
        most_errors = []
        for family in FAMILIES:
            members = [row for row in subset if row["family"] == family]
            if not members:
                continue
            family_x = _matrix(members, bits)
            true = _targets(members, MOST_TARGETS)
            pred = _predict_mean(most_models[family], family_x)
            most_errors.extend((true - pred).reshape(-1).tolist())
        result["most"] = {
            "mae": float(np.mean(np.abs(most_errors))) if most_errors else math.nan,
            "rmse": float(np.sqrt(np.mean(np.square(most_errors)))) if most_errors else math.nan,
        }
        results[split] = result
    return results


def _check_scaffold_leakage(rows: list[dict[str, str]]) -> None:
    split_scaffolds: dict[str, set[str]] = {}
    for row in rows:
        split_scaffolds.setdefault(row["split"], set()).add(row["scaffold"])
    names = sorted(split_scaffolds)
    for index, first in enumerate(names):
        for second in names[index + 1:]:
            overlap = split_scaffolds[first] & split_scaffolds[second]
            if overlap:
                raise ValueError(f"Scaffold leakage between {first} and {second}: {sorted(overlap)[:3]}")


def _family_medians(rows: Iterable[dict[str, str]]) -> dict[str, dict[str, float]]:
    rows = list(rows)
    medians: dict[str, dict[str, float]] = {}
    for family in FAMILIES:
        members = [row for row in rows if row["family"] == family]
        if not members:
            raise ValueError(f"No training references for family {family}")
        medians[family] = {
            key: float(median(float(row[key]) for row in members))
            for key in ("uvb_auc", "uva_auc", "energy_kj_mol", "specific_energy_wh_kg", "log_half_life_h")
        }
    return medians


def _train_bundle(rows: list[dict[str, str]], config: dict[str, Any], purpose: str) -> ReviewerBundle:
    _check_scaffold_leakage(rows)
    train = [row for row in rows if row["split"] == "train"]
    if not train:
        raise ValueError("Training split is empty")
    bits = int(config["reviewers"]["fingerprint_bits"])
    members = int(config["reviewers"]["ensemble_size"])
    base_seed = int(config["project"]["default_seed"]) + (0 if purpose == "reward" else 500_009)
    x = _matrix(train, bits)
    spectrum_models = _fit_members(purpose, members, base_seed + 11, x, _targets(train, SPECTRAL_TARGETS))
    safety_models = _fit_members(purpose, members, base_seed + 23, x, _targets(train, SAFETY_TARGETS))
    most_models: dict[str, list[Any]] = {}
    for offset, family in enumerate(FAMILIES):
        family_rows = [row for row in train if row["family"] == family]
        if len(family_rows) < 5:
            raise ValueError(f"Insufficient family-aware training data for {family}")
        most_models[family] = _fit_members(
            purpose, members, base_seed + 101 + offset,
            _matrix(family_rows, bits), _targets(family_rows, MOST_TARGETS),
        )
    metrics = _split_metrics(rows, bits, spectrum_models, safety_models, most_models)
    references = {
        "spectral_smiles": [row["smiles"] for row in train],
        "most_smiles_by_family": {family: [row["smiles"] for row in train if row["family"] == family] for family in FAMILIES},
        "family_medians": _family_medians(train),
        "training_data_hash": stable_hash("\n".join(sorted(row["candidate_id"] for row in train)), 32),
        "training_scaffolds": sorted({row["scaffold"] for row in train}),
    }
    return ReviewerBundle(purpose, bits, spectrum_models, safety_models, most_models, references, metrics)


def train_reviewers(config: dict[str, Any], data_path: str | Path, output_dir: str | Path) -> dict[str, Any]:
    rows = read_csv(data_path)
    if len(rows) > int(config["project"]["max_training_structures"]):
        raise ValueError("Reviewer training rows exceed configured cap")
    tiers = {row.get("evidence_tier") for row in rows}
    if config["execution"]["mode"] == "production" and tiers == {"synthetic_smoke_only"}:
        raise RuntimeError("Production mode refuses synthetic-only reviewer data")
    destination = Path(output_dir)
    reward = _train_bundle(rows, config, "reward")
    evaluator = _train_bundle(rows, config, "evaluator")
    reward_path = destination / "reward" / "reviewers.pkl"
    evaluator_path = destination / "evaluator" / "reviewers.pkl"
    reward.save(reward_path)
    evaluator.save(evaluator_path)
    metadata = {
        "schema_version": "1.0",
        "rows": len(rows),
        "evidence_tiers": sorted(str(item) for item in tiers),
        "feature_schema": [f"morgan_{i}" for i in range(reward.bits)] + list(DESCRIPTOR_NAMES) + [f"family_{item}" for item in FAMILIES],
        "reward": {"algorithm": "RandomForestRegressor ensemble", "path": str(reward_path.resolve()), "metrics": reward.metrics},
        "evaluator": {"algorithm": "ExtraTreesRegressor ensemble", "path": str(evaluator_path.resolve()), "metrics": evaluator.metrics},
        "independent_instances": reward_path.resolve() != evaluator_path.resolve(),
        "limitations": [
            "Synthetic smoke labels are not experimental observations.",
            "Applicability domains are fingerprint-neighbourhood proxies.",
            "Phototoxicity output is conservative triage, never proof of safety.",
        ],
    }
    destination.mkdir(parents=True, exist_ok=True)
    (destination / "model_cards.json").write_text(json.dumps(metadata, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return metadata
'''
_load_embedded_module('mostgen.reviewers', _source, PROJECT_ROOT / 'mostgen' / 'reviewers.py')
print('loaded mostgen.reviewers')


loaded mostgen.reviewers


In [11]:
_source = r'''from __future__ import annotations

import json
import math
from dataclasses import dataclass
from typing import Any, Sequence

from .chemistry import ChemistryError, descriptors, safety_veto, synthetic_accessibility
from .data import WAVELENGTHS
from .numerics import (
    beer_lambert_transmittance,
    critical_wavelength,
    interval_score,
    lower_confidence_bound,
    sigmoid,
    trapezoid_auc,
    upper_confidence_bound,
    weighted_geometric_mean,
)
from .reviewers import ReviewerBundle


def _auc_uncertainty(wavelengths: Sequence[float], std: Sequence[float], low: float, high: float) -> float:
    # Independent-bin propagation is a conservative-enough transparent proxy
    # for the smoke ensemble. Production models should retain member curves.
    selected = [(w, s) for w, s in zip(wavelengths, std) if low <= w <= high]
    if len(selected) < 2:
        return 0.0
    variance = 0.0
    for (x1, s1), (x2, s2) in zip(selected, selected[1:]):
        dx = x2 - x1
        variance += (0.5 * dx) ** 2 * (s1**2 + s2**2)
    return math.sqrt(variance)


def _lambda_uncertainty(wavelengths: Sequence[float], mean: Sequence[float], std: Sequence[float]) -> float:
    centre = critical_wavelength(wavelengths, mean)
    lower_curve = [max(0.0, value - spread) for value, spread in zip(mean, std)]
    upper_curve = [max(0.0, value + spread) for value, spread in zip(mean, std)]
    return max(abs(centre - critical_wavelength(wavelengths, lower_curve)), abs(centre - critical_wavelength(wavelengths, upper_curve)))


@dataclass
class ScoringContext:
    config: dict[str, Any]
    reviewers: ReviewerBundle

    def review_candidate(
        self, candidate: dict[str, Any], method_id: str, seed: int,
        raw_prediction: dict[str, Any] | None = None,
    ) -> dict[str, Any]:
        smiles = candidate["smiles"]
        charged = candidate["charged_smiles"]
        family = candidate["family"]
        veto = safety_veto(smiles, self.config)
        base: dict[str, Any] = {
            **candidate,
            "method_id": method_id,
            "seed": seed,
            "valid": not veto.veto,
            "psoralen_alert": veto.psoralen_alert,
            "known_phototoxic_match": veto.known_phototoxic_match,
            "psoralen_similarity": veto.psoralen_similarity,
            "psoralen_similarity_warning": veto.psoralen_similarity_warning,
            "reactive_alerts": ";".join(veto.reactive_alerts),
            "unsupported_elements": ";".join(veto.unsupported_elements),
            "safety_veto": veto.veto,
            "not_iso_certified": True,
            "selected": False,
            "reviewer_purpose": self.reviewers.purpose,
        }
        if veto.veto:
            base.update({
                "reward": 0.0,
                "reward_pre_diversity": 0.0,
                "joint_uv_pass": False,
                "most_pass": False,
                "safety_pass": False,
                "joint_pass": False,
                "phototoxicity_uncertain": True,
                "ad_spectral": False,
                "ad_most": False,
                "failure_reasons": ";".join(veto.reasons),
                "spectrum_json": "[]",
            })
            return base
        try:
            raw = raw_prediction if raw_prediction is not None else self.reviewers.predict(smiles, family)
        except (ChemistryError, ValueError) as exc:
            base.update({
                "valid": False, "reward": 0.0, "joint_uv_pass": False,
                "most_pass": False, "safety_pass": False, "joint_pass": False,
                "phototoxicity_uncertain": True, "ad_spectral": False,
                "ad_most": False, "failure_reasons": f"reviewer_error:{exc}",
                "spectrum_json": "[]",
            })
            return base

        spectrum_mean = [max(0.0, raw["spectrum"].mean[f"abs_{w}"]) for w in WAVELENGTHS]
        spectrum_std = [max(0.0, raw["spectrum"].std[f"abs_{w}"]) for w in WAVELENGTHS]
        uvb = trapezoid_auc(WAVELENGTHS, spectrum_mean, 290.0, 320.0)
        uva = trapezoid_auc(WAVELENGTHS, spectrum_mean, 320.0, 400.0)
        uvb_std = _auc_uncertainty(WAVELENGTHS, spectrum_std, 290.0, 320.0)
        uva_std = _auc_uncertainty(WAVELENGTHS, spectrum_std, 320.0, 400.0)
        lambda_c = critical_wavelength(WAVELENGTHS, spectrum_mean)
        lambda_std = _lambda_uncertainty(WAVELENGTHS, spectrum_mean, spectrum_std)
        z = float(self.config["reviewers"]["confidence_z"])
        uvb_lcb = lower_confidence_bound(uvb, uvb_std, z)
        uva_lcb = lower_confidence_bound(uva, uva_std, z)
        lambda_lcb = lower_confidence_bound(lambda_c, lambda_std, z)
        medians = raw["family_reference_medians"]
        joint_uv = (
            uvb_lcb >= medians["uvb_auc"]
            and uva_lcb >= medians["uva_auc"]
            and lambda_lcb >= float(self.config["reviewers"]["lambda_c_min_nm"])
        )

        most_mean, most_std = raw["most"].mean, raw["most"].std
        energy = most_mean["energy_kj_mol"]
        energy_std = most_std["energy_kj_mol"]
        energy_lcb = lower_confidence_bound(energy, energy_std, z)
        specific = most_mean["specific_energy_wh_kg"]
        specific_std = most_std["specific_energy_wh_kg"]
        half_log = most_mean["log_half_life_h"]
        half_log_std = most_std["log_half_life_h"]
        half_hours = 10.0 ** half_log
        half_low = 10.0 ** lower_confidence_bound(half_log, half_log_std, z)
        half_high = 10.0 ** upper_confidence_bound(half_log, half_log_std, z)
        half_window = self.config["reviewers"]["half_life_window_hours"]
        most_pass = (
            energy_lcb > 0.0
            and energy_lcb >= medians["energy_kj_mol"]
            and half_window[0] <= half_hours <= half_window[1]
        )

        safety_mean, safety_std = raw["safety"].mean, raw["safety"].std
        photo = min(1.0, max(0.0, safety_mean["phototoxicity_probability"]))
        photo_std = safety_std["phototoxicity_probability"]
        photo_upper = min(1.0, upper_confidence_bound(photo, photo_std, z))
        uncertain_band = self.config["reviewers"]["uncertain_probability_band"]
        photo_uncertain = (
            uncertain_band[0] <= photo <= uncertain_band[1]
            or (photo - z * photo_std) <= uncertain_band[1] <= photo_upper
        )
        kp = safety_mean["kp_log_cm_s"]
        kp_std = safety_std["kp_log_cm_s"]
        kp_upper = upper_confidence_bound(kp, kp_std, z)
        sa = synthetic_accessibility(smiles)
        ad_threshold = float(self.config["reviewers"]["ad_similarity_threshold"])
        ad_spectral = raw["ad_spectral_similarity"] >= ad_threshold
        ad_most = raw["ad_most_similarity"] >= ad_threshold
        safety_pass = (
            not photo_uncertain
            and photo_upper <= float(self.config["reviewers"]["phototoxicity_max_probability"])
            and kp_upper <= float(self.config["reviewers"]["kp_max_log_cm_s"])
            and sa <= 5.0
        )

        floor = float(self.config["reward"]["floor"])
        components = {
            "uvb": sigmoid(uvb_lcb, medians["uvb_auc"], max(0.5, medians["uvb_auc"] * 0.12)),
            "uva": sigmoid(uva_lcb, medians["uva_auc"], max(0.5, medians["uva_auc"] * 0.12)),
            "lambda_c": sigmoid(lambda_lcb, float(self.config["reviewers"]["lambda_c_min_nm"]), 5.0),
            "energy": sigmoid(energy_lcb, medians["energy_kj_mol"], max(2.0, medians["energy_kj_mol"] * 0.10)),
            "half_life": interval_score(half_log, math.log10(half_window[0]), math.log10(half_window[1]), 0.13),
            "phototoxicity": sigmoid(float(self.config["reviewers"]["phototoxicity_max_probability"]) - photo_upper, 0.0, 0.07),
            "permeation": sigmoid(float(self.config["reviewers"]["kp_max_log_cm_s"]) - kp_upper, 0.0, 0.25),
            "sa": sigmoid(5.0 - sa, 0.0, 0.75),
            "ad": min(1.0, min(raw["ad_spectral_similarity"], raw["ad_most_similarity"]) / ad_threshold),
        }
        weights = self.config["reward"]["weights"]
        reward = weighted_geometric_mean(components, weights, floor)
        stage_rewards = {
            "chemistry": 1.0,
            "spectrum": weighted_geometric_mean({k: components[k] for k in ("uvb", "uva", "lambda_c")}, weights, floor),
            "most": weighted_geometric_mean({k: components[k] for k in ("energy", "half_life")}, weights, floor),
            "safety": weighted_geometric_mean({k: components[k] for k in ("phototoxicity", "permeation", "sa", "ad")}, weights, floor),
        }
        failures = []
        if not joint_uv:
            failures.append("uv_joint_lcb_or_lambda")
        if not most_pass:
            failures.append("most_energy_or_half_life")
        if not ad_spectral:
            failures.append("spectral_out_of_domain")
        if not ad_most:
            failures.append("most_out_of_domain")
        if photo_uncertain:
            failures.append("phototoxicity_uncertain")
        elif photo_upper > float(self.config["reviewers"]["phototoxicity_max_probability"]):
            failures.append("phototoxicity_risk")
        if kp_upper > float(self.config["reviewers"]["kp_max_log_cm_s"]):
            failures.append("permeation_risk")
        if sa > 5.0:
            failures.append("sa_proxy")
        joint_pass = bool(joint_uv and most_pass and safety_pass and ad_spectral and ad_most)
        d = descriptors(smiles)
        base.update({
            "reference_uvb_median": medians["uvb_auc"], "reference_uva_median": medians["uva_auc"],
            "reference_energy_median": medians["energy_kj_mol"],
            "uvb_auc": uvb, "uvb_auc_uncertainty": uvb_std, "uvb_auc_lcb": uvb_lcb,
            "uva_auc": uva, "uva_auc_uncertainty": uva_std, "uva_auc_lcb": uva_lcb,
            "lambda_c_nm": lambda_c, "lambda_c_uncertainty": lambda_std, "lambda_c_lcb": lambda_lcb,
            "uvb_transmittance": beer_lambert_transmittance(spectrum_mean[:7], 1.0),
            "uva_transmittance": beer_lambert_transmittance(spectrum_mean[6:], 1.0),
            "energy_kj_mol": energy, "energy_uncertainty": energy_std, "energy_lcb": energy_lcb,
            "specific_energy_wh_kg": specific, "specific_energy_uncertainty": specific_std,
            "log_half_life_h": half_log, "half_life_uncertainty_log": half_log_std,
            "half_life_h": half_hours, "half_life_lcb_h": half_low, "half_life_ucb_h": half_high,
            "kp_log_cm_s": kp, "kp_uncertainty": kp_std, "kp_ucb": kp_upper,
            "phototoxicity_probability": photo, "phototoxicity_uncertainty": photo_std,
            "phototoxicity_ucb": photo_upper, "phototoxicity_uncertain": photo_uncertain,
            "similarity_D_A": raw["ad_spectral_similarity"], "similarity_D_B": raw["ad_most_similarity"],
            "ad_spectral": ad_spectral, "ad_most": ad_most,
            "sa_score": sa, "mol_wt": d["mol_wt"], "logp": d["logp"], "tpsa": d["tpsa"],
            "joint_uv_pass": joint_uv, "most_pass": most_pass, "safety_pass": safety_pass,
            "joint_pass": joint_pass, "reward": reward, "reward_pre_diversity": reward,
            "reward_components_json": json.dumps(components, sort_keys=True),
            "stage_rewards_json": json.dumps(stage_rewards, sort_keys=True),
            "failure_reasons": ";".join(failures),
            "spectrum_json": json.dumps([{"wavelength_nm": w, "absorbance": round(a, 6), "uncertainty": round(s, 6)} for w, a, s in zip(WAVELENGTHS, spectrum_mean, spectrum_std)]),
        })
        return base

    def review_candidates(self, candidates: list[dict[str, Any]], method_id: str, seed: int) -> list[dict[str, Any]]:
        safe_indices = []
        safe_molecules = []
        for index, candidate in enumerate(candidates):
            if not safety_veto(candidate["smiles"], self.config).veto:
                safe_indices.append(index)
                safe_molecules.append((candidate["smiles"], candidate["family"]))
        predictions = self.reviewers.predict_many(safe_molecules)
        prediction_by_index = dict(zip(safe_indices, predictions))
        return [
            self.review_candidate(candidate, method_id, seed, prediction_by_index.get(index))
            for index, candidate in enumerate(candidates)
        ]
'''
_load_embedded_module('mostgen.scoring', _source, PROJECT_ROOT / 'mostgen' / 'scoring.py')
print('loaded mostgen.scoring')


loaded mostgen.scoring


In [12]:
_source = r'''from __future__ import annotations

import json
import math
import random
import sys
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

from rdkit import DataStructs

from .chemistry import fingerprint
from .data import read_csv, write_csv
from .provenance import stable_hash
from .reviewers import ReviewerBundle
from .scoring import ScoringContext


METHODS = ("prior_random", "weighted_retraining", "libinvent_rl")


def _family_quotas(total: int, families: list[str]) -> dict[str, int]:
    base, remainder = divmod(total, len(families))
    return {family: base + (1 if index < remainder else 0) for index, family in enumerate(families)}


def _weighted_choice(rng: random.Random, rows: list[dict[str, Any]], weights: list[float]) -> dict[str, Any]:
    total = sum(weights)
    if total <= 0.0:
        return rng.choice(rows)
    point = rng.random() * total
    cumulative = 0.0
    for row, weight in zip(rows, weights):
        cumulative += weight
        if cumulative >= point:
            return row
    return rows[-1]


def _diversity_clusters(rows: list[dict[str, Any]], bits: int, threshold: float = 0.58) -> None:
    centroids = []
    counts: Counter[int] = Counter()
    for row in rows:
        fp = fingerprint(row["smiles"], bits)
        cluster = None
        if centroids:
            similarities = DataStructs.BulkTanimotoSimilarity(fp, centroids)
            best = max(range(len(similarities)), key=similarities.__getitem__)
            if similarities[best] >= threshold:
                cluster = best
        if cluster is None:
            cluster = len(centroids)
            centroids.append(fp)
        prior_count = counts[cluster]
        counts[cluster] += 1
        row["ecfp_cluster"] = cluster
        row["cluster_prior_count"] = prior_count


def _apply_diversity_penalty(rows: list[dict[str, Any]], config: dict[str, Any]) -> None:
    _diversity_clusters(rows, int(config["reviewers"]["fingerprint_bits"]))
    penalty = float(config["reward"]["diversity_penalty"])
    for row in rows:
        base = float(row.get("reward_pre_diversity", 0.0))
        row["reward"] = base / (1.0 + penalty * int(row["cluster_prior_count"]))


def _prior_candidates(rng: random.Random, pool: list[dict[str, Any]], count: int) -> list[dict[str, Any]]:
    return rng.sample(pool, min(count, len(pool)))


def _adaptive_candidates(
    method: str,
    rng: random.Random,
    pool: list[dict[str, Any]],
    count: int,
    context: ScoringContext,
    seed: int,
) -> list[dict[str, Any]]:
    remaining = list(pool)
    synthon_value: dict[str, float] = defaultdict(lambda: 0.5)
    synthon_visits: Counter[str] = Counter()
    selected: list[dict[str, Any]] = []
    warmup = min(max(8, count // 8), count)
    stage_names = [entry["name"] for entry in context.config["reward"]["stages"]]
    batch_size = min(24, max(4, count // 20))
    while remaining and len(selected) < count:
        pending = []
        pending_stages = []
        for _ in range(min(batch_size, count - len(selected), len(remaining))):
            progress = (len(selected) + len(pending)) / max(1, count)
            stage = stage_names[min(len(stage_names) - 1, int(progress * len(stage_names)))]
            if len(selected) + len(pending) < warmup:
                candidate = rng.choice(remaining)
            else:
                weights = []
                for row in remaining:
                    q = 0.5 * (synthon_value[row["synthon_a"]] + synthon_value[row["synthon_b"]])
                    if method == "weighted_retraining":
                        weight = 0.05 + q**3
                    else:
                        visits = 1 + synthon_visits[row["synthon_a"]] + synthon_visits[row["synthon_b"]]
                        bonus = math.sqrt(math.log(2 + len(selected) + len(pending)) / visits)
                        weight = 0.03 + math.exp(min(4.0, 2.2 * q + 0.35 * bonus))
                    weights.append(weight)
                candidate = _weighted_choice(rng, remaining, weights)
            remaining.remove(candidate)
            pending.append(candidate)
            pending_stages.append(stage)
        reviewed_batch = context.review_candidates(pending, method, seed)
        for reviewed, stage, candidate in zip(reviewed_batch, pending_stages, pending):
            reviewed["curriculum_stage"] = stage
            stage_rewards = json.loads(reviewed.get("stage_rewards_json", "{}") or "{}")
            signal = float(stage_rewards.get(stage, reviewed.get("reward_pre_diversity", 0.0)))
            for synthon in (candidate["synthon_a"], candidate["synthon_b"]):
                visits = synthon_visits[synthon]
                synthon_value[synthon] = (synthon_value[synthon] * visits + signal) / (visits + 1)
                synthon_visits[synthon] += 1
            selected.append(reviewed)
    return selected


def run_method(
    config: dict[str, Any],
    library_path: str | Path,
    reviewer_path: str | Path,
    method: str,
    seed: int,
    budget: int | None = None,
) -> list[dict[str, Any]]:
    if method not in METHODS:
        raise ValueError(f"Unknown search method: {method}")
    pool = read_csv(library_path)
    reviewers = ReviewerBundle.load(reviewer_path)
    context = ScoringContext(config, reviewers)
    calls = int(budget or config["execution"]["reviewer_budget_per_run"])
    if calls < int(config["execution"]["n_per_run"]):
        raise ValueError("Reviewer call budget cannot produce the requested unique count")
    families = sorted(config["families"])
    quotas = _family_quotas(calls, families)
    rng = random.Random(seed ^ int(stable_hash(method, 8), 16))
    results: list[dict[str, Any]] = []
    for family in families:
        members = [row for row in pool if row["family"] == family]
        quota = quotas[family]
        if len(members) < quota:
            raise ValueError(f"Family {family} has only {len(members)} candidates for quota {quota}")
        if method == "prior_random":
            candidates = _prior_candidates(rng, members, quota)
            results.extend(context.review_candidates(candidates, method, seed))
        else:
            results.extend(_adaptive_candidates(method, rng, members, quota, context, seed))
    _apply_diversity_penalty(results, config)
    results.sort(key=lambda row: (row["family"], -float(row.get("reward", 0.0)), row["smiles"]))
    for rank, row in enumerate(results, start=1):
        row["method_rank"] = rank
        row["reviewer_call"] = rank
    if len(results) != calls or len({row["smiles"] for row in results}) != len(results):
        raise RuntimeError("Search failed the exact-budget/uniqueness invariant")
    return results


def run_methods(
    config: dict[str, Any],
    library_path: str | Path,
    reviewer_path: str | Path,
    output_path: str | Path,
    methods: list[str],
) -> dict[str, Any]:
    all_rows: list[dict[str, Any]] = []
    run_counts: dict[str, int] = {}
    for method in methods:
        for seed in config["execution"]["seeds"]:
            rows = run_method(config, library_path, reviewer_path, method, int(seed))
            key = f"{method}:{seed}"
            run_counts[key] = len(rows)
            all_rows.extend(rows)
    write_csv(output_path, all_rows)
    return {
        "path": str(Path(output_path).resolve()),
        "rows": len(all_rows),
        "run_counts": run_counts,
        "methods": methods,
        "seeds": config["execution"]["seeds"],
    }


def write_generator_manifests(config: dict[str, Any], output_dir: str | Path) -> dict[str, Any]:
    destination = Path(output_dir)
    destination.mkdir(parents=True, exist_ok=True)
    manifests = []
    toml_configs = []
    root = destination.parent
    scorer = Path(__file__).resolve().parents[1] / "scripts" / "reinvent_external_score.py"
    prior = config["production"]["reaction_prior_path"] or "PRIOR_PATH_REQUIRED"
    per_family_budget = math.ceil(int(config["execution"]["reviewer_budget_per_run"]) / len(config["families"]))
    steps_per_stage = 2
    batch_size = max(1, math.ceil(per_family_budget / (4 * steps_per_stage)))
    for family, family_config in config["families"].items():
        scaffold = family_config["ground_template"].format(r1="[*:1]", r2="[*:2]")
        scaffold_path = destination / f"scaffold_{family}.smi"
        scaffold_path.write_text(scaffold + "\n", encoding="utf-8")
        model_path = root / "models" / "reward" / "reviewers.pkl"
        library_path = root / "data" / "reaction_library.csv"
        lines = [
            'run_type = "staged_learning"',
            f'device = "{"cpu" if config["execution"]["mode"] == "smoke" else "cuda:0"}"',
            f'tb_logdir = "{(destination / ("tb_" + family)).as_posix()}"',
            f'json_out_config = "{(destination / ("resolved_" + family + ".json")).as_posix()}"',
            "", "[parameters]",
            f'prior_file = "{prior}"', f'agent_file = "{prior}"',
            f'smiles_file = "{scaffold_path.resolve().as_posix()}"',
            f'summary_csv_prefix = "{(destination / ("rl_" + family)).as_posix()}"',
            f"batch_size = {batch_size}", "randomize_smiles = true", "use_checkpoint = false", "purge_memories = false",
            "", "[learning_strategy]", 'type = "dap"', "sigma = 128", "rate = 0.0001",
            "", "[diversity_filter]", 'type = "PenalizeSameSmiles"', "bucket_size = 25", "minscore = 0.35", "penalty_multiplier = 0.5",
        ]
        for index, stage in enumerate(config["reward"]["stages"], start=1):
            stage_name = stage["name"]
            args = (
                f'{scorer.resolve().as_posix()} --config {Path(config["_config_path"]).resolve().as_posix()} '
                f'--model {model_path.resolve().as_posix()} --library {library_path.resolve().as_posix()} '
                f'--family {family} --stage {stage_name}'
            )
            lines.extend([
                "", "[[stage]]", f'chkpt_file = "{(destination / (family + "_stage" + str(index) + ".chkpt")).as_posix()}"',
                'termination = "simple"', "max_score = 0.999", f"min_steps = {steps_per_stage}", f"max_steps = {steps_per_stage}",
                "", "[stage.scoring]", 'type = "geometric_mean"',
                "", "[[stage.scoring.component]]", "[stage.scoring.component.ExternalProcess]",
                "[[stage.scoring.component.ExternalProcess.endpoint]]", f'name = "MOSTGen {stage_name}"', "weight = 1.0",
                f'params.executable = "{Path(sys.executable).resolve().as_posix()}"', f'params.args = "{args}"',
                'params.property = "stage_score"',
            ])
        toml_path = destination / f"reinvent4_{family}.toml"
        toml_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
        toml_configs.append(str(toml_path.resolve()))
        manifest = {
            "schema_version": "1.0",
            "family": family,
            "generator": "REINVENT4 LibInvent",
            "backend_status": "surrogate_smoke" if config["execution"]["backend"] == "surrogate_smoke" else "external_required",
            "repository": config["production"]["reinvent4_repository"],
            "revision": config["production"]["reinvent4_revision"],
            "version": config["production"]["reinvent4_version"],
            "prior": config["production"]["reaction_prior_path"] or "NOT_CONFIGURED",
            "reaction_smarts": family_config["reaction_smarts"],
            "allowed_positions": family_config["allowed_positions"],
            "curriculum": config["reward"]["stages"],
            "reward_weights": config["reward"]["weights"],
            "diversity_filter": {"type": "ECFP centroid", "penalty": config["reward"]["diversity_penalty"]},
            "seed_runs": config["execution"]["seeds"],
            "reviewer_budget_per_run": config["execution"]["reviewer_budget_per_run"],
            "toml_config": str(toml_path.resolve()),
            "scaffold_file": str(scaffold_path.resolve()),
            "external_scorer": str(scorer.resolve()),
        }
        path = destination / f"reinvent4_{family}.json"
        path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")
        manifests.append(str(path.resolve()))
    policy = {
        "status": "manifest_only" if config["execution"]["backend"] == "surrogate_smoke" else "ready_for_external_validation",
        "family_manifests": manifests,
        "reinvent_toml_configs": toml_configs,
        "reinvent_command": "reinvent -l <family>.log <family>.toml",
        "note": "The smoke policy is an adaptive reaction-library search, not a trained REINVENT neural prior.",
    }
    (destination / "generator_policy.json").write_text(json.dumps(policy, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return policy
'''
_load_embedded_module('mostgen.search', _source, PROJECT_ROOT / 'mostgen' / 'search.py')
print('loaded mostgen.search')


loaded mostgen.search


In [13]:
_source = r'''from __future__ import annotations

import json
import math
import random
from collections import Counter, defaultdict
from pathlib import Path
from statistics import fmean
from typing import Any, Callable

from rdkit import DataStructs

from .chemistry import fingerprint, murcko_scaffold
from .data import read_csv, write_csv


def _truth(value: Any) -> bool:
    return value is True or str(value).lower() in {"true", "1", "yes"}


def _float(row: dict[str, Any], key: str, default: float = 0.0) -> float:
    try:
        return float(row.get(key, default))
    except (TypeError, ValueError):
        return default


def _internal_diversity(rows: list[dict[str, Any]], bits: int, seed: int = 701) -> float:
    unique = sorted({row["smiles"] for row in rows})
    if len(unique) < 2:
        return 0.0
    rng = random.Random(seed)
    pairs = []
    max_pairs = min(500, len(unique) * (len(unique) - 1) // 2)
    seen = set()
    while len(pairs) < max_pairs:
        left, right = sorted(rng.sample(range(len(unique)), 2))
        if (left, right) in seen:
            continue
        seen.add((left, right))
        pairs.append((left, right))
    fps = [fingerprint(smiles, bits) for smiles in unique]
    return fmean(1.0 - float(DataStructs.TanimotoSimilarity(fps[left], fps[right])) for left, right in pairs)


def _effective_sample_size(weights: list[float]) -> float:
    total = sum(weights)
    squares = sum(value * value for value in weights)
    return total * total / squares if squares else 0.0


def _run_metrics(rows: list[dict[str, Any]], training_smiles: set[str], bits: int) -> dict[str, Any]:
    n = len(rows)
    unique = {row["smiles"] for row in rows}
    valid = [row for row in rows if _truth(row.get("valid"))]
    scaffolds = {murcko_scaffold(row["smiles"]) for row in valid}
    families = Counter(row["family"] for row in valid)
    rewards = [_float(row, "reward") for row in rows]
    return {
        "evaluated": n,
        "validity": len(valid) / n if n else 0.0,
        "uniqueness": len(unique) / n if n else 0.0,
        "novelty": sum(row["smiles"] not in training_smiles for row in valid) / len(valid) if valid else 0.0,
        "internal_diversity": _internal_diversity(valid, bits),
        "scaffold_diversity": len(scaffolds) / len(valid) if valid else 0.0,
        "mean_sa_score": fmean(_float(row, "sa_score", 10.0) for row in valid) if valid else 10.0,
        "joint_success": sum(_truth(row.get("joint_pass")) for row in rows) / n if n else 0.0,
        "both_ad_fraction": sum(_truth(row.get("ad_spectral")) and _truth(row.get("ad_most")) for row in rows) / n if n else 0.0,
        "family_coverage": sum(1 for family in families if families[family] > 0) / 3.0,
        "nbd_qc_count": families["nbd_qc"],
        "dewar_pyrimidinone_count": families["dewar_pyrimidinone"],
        "spiropyran_count": families["spiropyran"],
        "nonzero_reward_fraction": sum(value > 0.0 for value in rewards) / n if n else 0.0,
        "effective_sample_size": _effective_sample_size(rewards),
        "unique_ecfp_clusters": len({row.get("ecfp_cluster") for row in rows}),
    }


def _bootstrap(values: list[float], samples: int, seed: int) -> tuple[float, float, float]:
    if not values:
        return math.nan, math.nan, math.nan
    if len(values) == 1:
        return values[0], values[0], values[0]
    rng = random.Random(seed)
    means = []
    for _ in range(samples):
        means.append(fmean(rng.choice(values) for _ in values))
    means.sort()
    low = means[max(0, int(0.025 * len(means)) - 1)]
    high = means[min(len(means) - 1, int(0.975 * len(means)))]
    return fmean(values), low, high


def _ablation_rows(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    result = []
    grouped: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for row in rows:
        grouped[row["method_id"]].append(row)
    for method, members in sorted(grouped.items()):
        gates: dict[str, Callable[[dict[str, Any]], bool]] = {
            "full": lambda r: _truth(r.get("joint_pass")),
            "without_uncertainty_penalty": lambda r: (
                _float(r, "uvb_auc") >= _float(r, "reference_uvb_median")
                and _float(r, "uva_auc") >= _float(r, "reference_uva_median")
                and _float(r, "lambda_c_nm") >= 370.0
                and _float(r, "energy_kj_mol") >= _float(r, "reference_energy_median")
                and 4.0 <= _float(r, "half_life_h") <= 24.0
                and _truth(r.get("safety_pass")) and _truth(r.get("ad_spectral")) and _truth(r.get("ad_most"))
            ),
            "without_safety_gate": lambda r: _truth(r.get("joint_uv_pass")) and _truth(r.get("most_pass")) and _truth(r.get("ad_spectral")) and _truth(r.get("ad_most")),
            "without_ad_gate": lambda r: _truth(r.get("joint_uv_pass")) and _truth(r.get("most_pass")) and _truth(r.get("safety_pass")),
        }
        for setting, predicate in gates.items():
            passed = sum(predicate(row) for row in members)
            result.append({"method_id": method, "ablation": setting, "evaluated": len(members), "eligible": passed, "success_fraction": passed / len(members) if members else 0.0})
        top_adjusted = sorted(members, key=lambda row: -_float(row, "reward"))[: min(100, len(members))]
        top_raw = sorted(members, key=lambda row: -_float(row, "reward_pre_diversity"))[: min(100, len(members))]
        result.append({
            "method_id": method, "ablation": "without_diversity_filter",
            "evaluated": len(members), "eligible": sum(_truth(row.get("joint_pass")) for row in top_raw),
            "success_fraction": sum(_truth(row.get("joint_pass")) for row in top_raw) / len(top_raw) if top_raw else 0.0,
            "top100_cluster_diversity_full": len({row.get("ecfp_cluster") for row in top_adjusted}) / len(top_adjusted) if top_adjusted else 0.0,
            "top100_cluster_diversity_ablated": len({row.get("ecfp_cluster") for row in top_raw}) / len(top_raw) if top_raw else 0.0,
        })
    return result


def compute_metrics(
    generated_path: str | Path,
    training_path: str | Path,
    output_dir: str | Path,
    config: dict[str, Any],
) -> dict[str, Any]:
    rows = read_csv(generated_path)
    training_smiles = {row["smiles"] for row in read_csv(training_path)}
    grouped: dict[tuple[str, str], list[dict[str, Any]]] = defaultdict(list)
    for row in rows:
        grouped[(row["method_id"], row["seed"])].append(row)
    run_rows = []
    bits = int(config["reviewers"]["fingerprint_bits"])
    for (method, seed), members in sorted(grouped.items()):
        run_rows.append({"method_id": method, "seed": int(seed), **_run_metrics(members, training_smiles, bits)})
    summary_rows = []
    metric_names = ("validity", "uniqueness", "novelty", "internal_diversity", "scaffold_diversity", "mean_sa_score", "joint_success", "both_ad_fraction", "family_coverage")
    for method in sorted({row["method_id"] for row in run_rows}):
        members = [row for row in run_rows if row["method_id"] == method]
        summary: dict[str, Any] = {"method_id": method, "seed_runs": len(members)}
        for metric in metric_names:
            mean, low, high = _bootstrap([float(row[metric]) for row in members], int(config["execution"]["bootstrap_samples"]), int(config["project"]["default_seed"]) ^ sum(map(ord, method + metric)))
            summary[f"{metric}_mean"] = mean
            summary[f"{metric}_ci_low"] = low
            summary[f"{metric}_ci_high"] = high
        summary_rows.append(summary)
    diagnostics = []
    for (method, seed), members in sorted(grouped.items()):
        component_values: dict[str, list[float]] = defaultdict(list)
        for row in members:
            try:
                parsed = json.loads(row.get("reward_components_json", "{}"))
            except json.JSONDecodeError:
                parsed = {}
            for key, value in parsed.items():
                component_values[key].append(float(value))
        rewards = [_float(row, "reward") for row in members]
        diagnostics.append({
            "method_id": method, "seed": seed,
            "nonzero_fraction": sum(value > 0 for value in rewards) / len(rewards) if rewards else 0.0,
            "effective_sample_size": _effective_sample_size(rewards),
            "largest_cluster_fraction": max(Counter(row.get("ecfp_cluster") for row in members).values()) / len(members) if members else 0.0,
            **{f"component_{key}_mean": fmean(values) for key, values in sorted(component_values.items())},
        })
    destination = Path(output_dir)
    destination.mkdir(parents=True, exist_ok=True)
    write_csv(destination / "metrics_runs.csv", run_rows)
    write_csv(destination / "metrics_summary.csv", summary_rows)
    write_csv(destination / "ablations.csv", _ablation_rows(rows))
    write_csv(destination / "reward_diagnostics.csv", diagnostics)
    result = {"run_metrics": run_rows, "summary": summary_rows, "matched_reviewer_budget": len({len(group) for group in grouped.values()}) == 1}
    (destination / "metrics.json").write_text(json.dumps(result, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return result
'''
_load_embedded_module('mostgen.metrics', _source, PROJECT_ROOT / 'mostgen' / 'metrics.py')
print('loaded mostgen.metrics')


loaded mostgen.metrics


In [14]:
_source = r'''from __future__ import annotations

import csv
import json
import shutil
from pathlib import Path
from typing import Any, Iterable

from rdkit import Chem
from rdkit.Chem import AllChem

from .data import write_csv
from .provenance import stable_hash


def availability(config: dict[str, Any]) -> dict[str, Any]:
    tools = {
        "xtb": shutil.which(config["production"]["xtb_executable"]),
        "stda": shutil.which(config["production"]["stda_executable"]),
        "xtb4stda": shutil.which(config["production"].get("xtb4stda_executable", "xtb4stda")),
    }
    return {"tools": tools, "ready": all(tools.values())}


def _write_low_energy_xyz(smiles: str, path: Path, seed: int) -> dict[str, Any]:
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    params = AllChem.ETKDGv3()
    params.randomSeed = int(seed & 0x7FFFFFFF)
    conformers = list(AllChem.EmbedMultipleConfs(mol, numConfs=8, params=params))
    if not conformers:
        raise RuntimeError("RDKit conformer embedding failed")
    energies = []
    for conf_id in conformers:
        try:
            if AllChem.MMFFHasAllMoleculeParams(mol):
                props = AllChem.MMFFGetMoleculeProperties(mol)
                forcefield = AllChem.MMFFGetMoleculeForceField(mol, props, confId=conf_id)
            else:
                forcefield = AllChem.UFFGetMoleculeForceField(mol, confId=conf_id)
            forcefield.Minimize(maxIts=500)
            energies.append((float(forcefield.CalcEnergy()), conf_id))
        except Exception:
            continue
    if not energies:
        raise RuntimeError("No conformer could be force-field minimized")
    energy, best = min(energies)
    Chem.MolToXYZFile(mol, str(path), confId=int(best))
    return {"conformers": len(conformers), "selected_forcefield_energy": energy}


def prepare_oracle_queue(
    rows: Iterable[dict[str, Any]],
    output_dir: str | Path,
    config: dict[str, Any],
) -> dict[str, Any]:
    destination = Path(output_dir)
    structures = destination / "structures"
    structures.mkdir(parents=True, exist_ok=True)
    status = availability(config)
    queue = []
    errors = []
    for rank, row in enumerate(rows, start=1):
        candidate_id = row.get("candidate_id") or stable_hash(row["smiles"], 20)
        try:
            ground_path = structures / f"{candidate_id}_ground.xyz"
            charged_path = structures / f"{candidate_id}_charged.xyz"
            ground_info = _write_low_energy_xyz(row["smiles"], ground_path, int(config["project"]["default_seed"]) + rank)
            charged_info = _write_low_energy_xyz(row["charged_smiles"], charged_path, int(config["project"]["default_seed"]) + 100_000 + rank)
            queue.append({
                "oracle_rank": rank, "candidate_id": candidate_id, "family": row["family"],
                "smiles": row["smiles"], "charged_smiles": row["charged_smiles"],
                "ground_xyz": str(ground_path.resolve()), "charged_xyz": str(charged_path.resolve()),
                "ground_conformers": ground_info["conformers"], "charged_conformers": charged_info["conformers"],
                "xtb_energy_command_ground": f"{config['production']['xtb_executable']} {ground_path.name} --gfn 2 --opt tight --json",
                "xtb_energy_command_charged": f"{config['production']['xtb_executable']} {charged_path.name} --gfn 2 --opt tight --json",
                "spectral_commands": "xtb4stda <xyz> then stda -xtb; broaden transitions over 290-400 nm",
                "oracle_status": "queued_external" if status["ready"] else "not_run_tools_unavailable",
            })
        except Exception as exc:
            errors.append({"candidate_id": candidate_id, "error": str(exc)})
    write_csv(destination / "oracle_queue.csv", queue)
    manifest = {
        "schema_version": "1.0", "availability": status, "queued": len(queue), "errors": errors,
        "calculation_contract": {
            "energy": "independent GFN2-xTB optimization and ground/charged energy difference",
            "spectrum": "sTDA-xTB transitions broadened over 290-400 nm",
            "inside_rl_loop": False,
            "automatic_proxy_substitution": False,
        },
        "status": "queued_external" if status["ready"] else "not_run_external_tools_unavailable",
    }
    (destination / "oracle_manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return manifest
'''
_load_embedded_module('mostgen.oracle', _source, PROJECT_ROOT / 'mostgen' / 'oracle.py')
print('loaded mostgen.oracle')


loaded mostgen.oracle


In [15]:
_source = r'''from __future__ import annotations

import json
from collections import defaultdict
from pathlib import Path
from typing import Any

from .data import read_csv, write_csv
from .oracle import prepare_oracle_queue
from .reviewers import ReviewerBundle
from .scoring import ScoringContext


def _truth(value: Any) -> bool:
    return value is True or str(value).lower() in {"true", "1", "yes"}


def _balanced_top(rows: list[dict[str, str]], count: int) -> list[dict[str, str]]:
    deduplicated: dict[str, dict[str, str]] = {}
    for row in sorted(rows, key=lambda item: -float(item.get("reward", 0.0))):
        deduplicated.setdefault(row["smiles"], row)
    by_family: dict[str, list[dict[str, str]]] = defaultdict(list)
    for row in deduplicated.values():
        if not _truth(row.get("safety_veto")):
            by_family[row["family"]].append(row)
    families = sorted(by_family)
    base, remainder = divmod(count, max(1, len(families)))
    chosen = []
    for index, family in enumerate(families):
        chosen.extend(by_family[family][: base + (1 if index < remainder else 0)])
    if len(chosen) < count:
        used = {row["smiles"] for row in chosen}
        extras = [row for row in sorted(deduplicated.values(), key=lambda item: -float(item.get("reward", 0.0))) if row["smiles"] not in used]
        chosen.extend(extras[: count - len(chosen)])
    return chosen[:count]


def _card(row: dict[str, Any], oracle_status: str) -> dict[str, Any]:
    try:
        spectrum = json.loads(row.get("spectrum_json", "[]"))
    except json.JSONDecodeError:
        spectrum = []
    return {
        "candidate": {
            "candidate_id": row.get("candidate_id"), "smiles": row["smiles"],
            "charged_smiles": row["charged_smiles"], "family": row["family"],
            "method_id": row["method_id"], "seed": row["seed"],
        },
        "spectrum": {
            "curve": spectrum, "uvb_auc": row.get("uvb_auc"), "uva_auc": row.get("uva_auc"),
            "lambda_c_proxy_nm": row.get("lambda_c_nm"), "uncertainty": {
                "uvb_auc": row.get("uvb_auc_uncertainty"), "uva_auc": row.get("uva_auc_uncertainty"),
                "lambda_c": row.get("lambda_c_uncertainty"),
            },
        },
        "most": {
            "delta_h_kj_mol": row.get("energy_kj_mol"), "specific_energy_wh_kg": row.get("specific_energy_wh_kg"),
            "half_life_h_at_305k": row.get("half_life_h"), "uncertainty": {
                "delta_h": row.get("energy_uncertainty"), "specific_energy": row.get("specific_energy_uncertainty"),
                "log_half_life": row.get("half_life_uncertainty_log"),
            },
        },
        "safety_triage": {
            "kp_log_cm_s": row.get("kp_log_cm_s"), "phototoxicity_probability": row.get("phototoxicity_probability"),
            "phototoxicity_uncertain": row.get("phototoxicity_uncertain"), "psoralen_alert": row.get("psoralen_alert"),
            "known_phototoxic_match": row.get("known_phototoxic_match"), "reactive_alerts": row.get("reactive_alerts"),
            "sa_score_proxy": row.get("sa_score"),
        },
        "applicability_domain": {
            "spectral_similarity_D_A": row.get("similarity_D_A"), "most_similarity_D_B": row.get("similarity_D_B"),
            "spectral_inside": row.get("ad_spectral"), "most_inside": row.get("ad_most"),
        },
        "decision": {
            "uv_pass": row.get("joint_uv_pass"), "most_pass": row.get("most_pass"),
            "safety_pass": row.get("safety_pass"), "joint_pass": row.get("joint_pass"),
            "failure_reasons": row.get("failure_reasons"), "physical_oracle": oracle_status,
            "selected": row.get("selected", False),
        },
        "evidence": {
            "reviewer": "independent evaluator, separate from reward models",
            "data_tier": "synthetic_smoke_only",
            "physical_oracle": oracle_status,
            "claim_level": "research screening candidate only",
            "not_iso_certified": True,
            "experimental_phototoxicity_required": "OECD TG 432 or suitable successor method",
        },
    }


def review_generated(
    config: dict[str, Any],
    generated_path: str | Path,
    evaluator_path: str | Path,
    output_dir: str | Path,
) -> dict[str, Any]:
    generated = read_csv(generated_path)
    top = _balanced_top(generated, int(config["execution"]["shortlist_size"]))
    evaluators = ReviewerBundle.load(evaluator_path)
    context = ScoringContext(config, evaluators)
    reviewed = []
    for row in top:
        evaluated = context.review_candidate(row, row["method_id"], int(row["seed"]))
        evaluated["reward_model_reward"] = float(row.get("reward", 0.0))
        evaluated["preselection_rank"] = len(reviewed) + 1
        reviewed.append(evaluated)
    destination = Path(output_dir)
    destination.mkdir(parents=True, exist_ok=True)
    oracle_manifest = prepare_oracle_queue(reviewed, destination / "physical_oracle", config)
    oracle_complete = oracle_manifest["status"] == "complete"
    provisional = [
        row for row in reviewed
        if _truth(row.get("joint_pass"))
        and not _truth(row.get("phototoxicity_uncertain"))
        and not _truth(row.get("psoralen_alert"))
        and not _truth(row.get("known_phototoxic_match"))
    ]
    selected = provisional if oracle_complete else []
    selected_keys = {(row["smiles"], row["method_id"], str(row["seed"])) for row in selected}
    for row in reviewed:
        row["physical_oracle_status"] = oracle_manifest["status"]
        row["selected"] = (row["smiles"], row["method_id"], str(row["seed"])) in selected_keys
        row["selection_status"] = "selected_research_candidate" if row["selected"] else (
            "provisional_pending_physical_oracle" if row in provisional else "failed_independent_review"
        )
    for row in generated:
        row["selected"] = (row["smiles"], row["method_id"], str(row["seed"])) in selected_keys
    write_csv(generated_path, generated)
    write_csv(destination / "reviewed_top.csv", reviewed)
    write_csv(destination / "provisional_shortlist.csv", provisional, fieldnames=list(reviewed[0]) if reviewed else [])
    write_csv(destination / "shortlist.csv", selected, fieldnames=list(reviewed[0]) if reviewed else [])
    cards_dir = destination / "cards"
    cards_dir.mkdir(parents=True, exist_ok=True)
    for row in reviewed:
        card = _card(row, oracle_manifest["status"])
        (cards_dir / f"{row['candidate_id']}.json").write_text(json.dumps(card, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    summary = {
        "reviewed": len(reviewed), "independent_joint_pass": len(provisional),
        "selected_after_physical_oracle": len(selected), "physical_oracle": oracle_manifest,
        "selection_policy": "Fail closed: no final selection until independent physical oracle results are complete.",
    }
    (destination / "review_summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return summary

'''
_load_embedded_module('mostgen.review', _source, PROJECT_ROOT / 'mostgen' / 'review.py')
print('loaded mostgen.review')


loaded mostgen.review


In [16]:
_source = r'''from __future__ import annotations

import json
from pathlib import Path
from typing import Any

from .data import read_csv


def _pct(value: Any) -> str:
    try:
        return f"{100.0 * float(value):.2f}%"
    except (TypeError, ValueError):
        return "n/a"


def build_reports(experiment_dir: str | Path, config: dict[str, Any]) -> dict[str, str]:
    root = Path(experiment_dir)
    output = root / "report"
    output.mkdir(parents=True, exist_ok=True)
    metrics_path = root / "metrics" / "metrics_summary.csv"
    metrics = read_csv(metrics_path) if metrics_path.exists() else []
    generated = read_csv(root / "generated.csv")
    review = json.loads((root / "review" / "review_summary.json").read_text(encoding="utf-8"))
    table = ["| Метод | Валидность | Уникальность | Diversity | Joint success | Обе AD |", "|---|---:|---:|---:|---:|---:|"]
    for row in metrics:
        table.append(
            f"| {row['method_id']} | {_pct(row['validity_mean'])} | {_pct(row['uniqueness_mean'])} | "
            f"{float(row['internal_diversity_mean']):.3f} | {_pct(row['joint_success_mean'])} | {_pct(row['both_ad_fraction_mean'])} |"
        )
    family_counts = {family: sum(row["family"] == family for row in generated) for family in config["families"]}
    no_joint = sum(str(row.get("joint_pass", "")).lower() == "true" for row in generated) == 0
    report = f"""# Воспроизводимый скрининг UV-поглощающих MOST-фотопереключателей

## 1. Цель, область утверждений и вычислительный бюджет

Цель прототипа — найти вычислительное пересечение широкополосного поглощения 290–400 нм, молекулярного накопления энергии, времени хранения 4–24 ч при 305 K и консервативного safety-triage. Проектируется одна малая фотопереключаемая молекула, не смесь и не готовая солнцезащитная формуляция. Результаты имеют статус **screening proxy**. Они не подтверждают безопасность, эффективность, фотостабильность или пригодность вещества как косметического ингредиента.

ISO 24444:2019 описывает in vivo определение SPF готового продукта, а ISO 24443:2021 — in vitro оценку UVA-защиты продукта. Поэтому ни одна молекула здесь не называется соответствующей ISO. Следующий уровень доказательности требует изготовления плёнки/формуляции и испытания продукта. Фототоксичность требует отдельной экспериментальной проверки, например OECD TG 432.

Запуск: режим `{config['execution']['mode']}`, backend `{config['execution']['backend']}`, seeds `{config['execution']['seeds']}`. Лимиты: не более {config['project']['max_training_structures']} обучающих структур и {config['project']['max_gpu_hours']:.1f} GPU·ч; фактическая конфигурация заявляет {config['execution']['gpu_hours']:.1f} GPU·ч. Каждый метод получил ровно {config['execution']['reviewer_budget_per_run']} вызовов reward-reviewer на seed.

## 2. Генеративное пространство и сравниваемые методы

Три семейства запускались как family-aware пространства с фиксированными реакционными каркасами и двумя разрешёнными позициями замещения: NBD/QC как основной физически определённый класс, Dewar-pyrimidinone как малодатовый экспериментальный класс и spiropyran/merocyanine как проверка переносимости. Заряженный изомер строился детерминированно; RDKit проверял валентность, санитаризацию и равенство молекулярной формулы пары. Размеры фактически оценённых подмножеств: `{family_counts}`.

Сравнены (i) случайный reaction-constrained prior, (ii) adaptive weighted-retraining по той же библиотеке синтонов и (iii) curriculum-поиск, являющийся CPU-заместителем LibInvent RL. Для production hand-off созданы отдельные REINVENT4/LibInvent manifests по семействам. Smoke backend не является нейросетевым REINVENT prior и не должен так интерпретироваться.

Curriculum последовательно открывает валидность/семейство/пару, спектр, MOST, затем безопасность/SA/AD. Награда — взвешенное геометрическое среднее непрерывных sigmoid/interval-компонентов с floor `{config['reward']['floor']}`. ECFP-centroid filter штрафует повторное заселение кластеров. Таблица `reward_diagnostics.csv` позволяет проверить долю ненулевых наград, effective sample size, баланс компонентов и scaffold collapse.

## 3. Reviewer-модели, неопределённость и применимость

Pipeline сохраняет scaffold split до обучения и проверяет отсутствие пересечения scaffold между train/validation/test. Reward-модели и независимые evaluator-модели сериализованы раздельно: первые используют ensemble Random Forest и направляют поиск, вторые — отдельный ensemble Extra Trees и переоценивают только top-кандидатов. MOST-регрессии разделены по семействам. Spectrum reviewer предсказывает сетку 290–400 нм с шагом 5 нм; из неё интегрируются UVB AUC, UVA AUC, critical-wavelength proxy и условная Beer–Lambert transmittance.

Joint UV-pass требует, чтобы нижняя 90%-я доверительная граница обоих AUC была не хуже медианы соответствующего семейства и чтобы нижняя граница λc-proxy была не ниже {config['reviewers']['lambda_c_min_nm']:.0f} нм. MOST-pass требует положительную нижнюю границу ΔH, результат не хуже семейной медианы и средний t½ в окне {config['reviewers']['half_life_window_hours'][0]:.0f}–{config['reviewers']['half_life_window_hours'][1]:.0f} ч. Высокий прогноз вне fingerprint-domain не засчитывается.

В текущем демонстрационном запуске метки reviewer-набора имеют evidence tier `synthetic_smoke_only`. Они создают проверяемую задачу для программного контура, но не заменяют M01/M03/M04/M11–M13 или U07–U17. Production mode специально отказывается обучаться на synthetic-only данных.

## 4. Safety gate и правила shortlist

До вычисления награды применяются SMARTS-поиск линейных и угловых фурокумариновых/псораленовых ядер, точное совпадение с локальным списком известных фототоксичных структур, запрет неподдерживаемых элементов, reactive/unstable alerts, проверка валентности и построения изомерной пары. Псоралены и иные фурокумарины не могут пройти в итоговый CSV. Это консервативно согласуется с заключением SCCP: безопасность фурокумаринов не была подтверждена, а фототоксичность нельзя считать исключённой.

Вероятность фототоксичности и Kp — лишь triage. Интервал неопределённости фототоксичности блокирует продвижение независимо от среднего значения. Отсутствие алерта не является доказательством безопасности. В карточке каждого top-кандидата раздельно показаны spectrum/MOST/safety, uncertainty, обе AD, SA-proxy, причины отказа и уровень доказательности.

## 5. Результаты и matched-budget comparison

{chr(10).join(table)}

Сгенерировано {len(generated)} строк; независимый evaluator пересмотрел {review['reviewed']}, provisional joint-pass получили {review['independent_joint_pass']}. После физического oracle выбрано {review['selected_after_physical_oracle']}. Статус oracle: `{review['physical_oracle']['status']}`. При отсутствии xTB/sTDA pipeline не подставляет суррогатный «квантовый» результат и оставляет `selected=false` — это намеренное fail-closed поведение.

Bootstrap-интервалы агрегируют три seed-запуска в full mode. `ablations.csv` показывает эффекты снятия uncertainty, safety, AD и diversity gates; сравнение curriculum с prior следует читать как matched-budget методическое сравнение, а не как доказательство превосходства на реальных данных.

## 6. Вывод, ограничения и следующий эксперимент

{'В этом запуске не найдено кандидатов, прошедших исходный joint gate. Это допустимый научный результат: таблицы причин отказа показывают конфликт требований и неопределённость; он не доказывает физическую невозможность.' if no_joint else 'Некоторые структуры прошли внутренний reward gate, но они остаются вычислительными гипотезами и требуют независимого физического и экспериментального подтверждения.'}

Следующий этап: подключить лицензированные/открытые записи M01/M03/M04 и M11–M13 с единицами, состояниями и условиями; U07–U17 — с endpoint-specific splits; зафиксировать commit REINVENT4 и checksum reaction prior; затем выполнить GFN2-xTB conformer review и sTDA-xTB broadened spectrum для 50–100 top-кандидатов. После синтеза нужны проверка идентичности обоих изомеров, циклируемость, quantum yield, растворитель/плёнка, OECD TG 432 и тестирование готовой формуляции по применимым стандартам.

## Приложение A. Протокол воспроизводимости и аудит данных

Единицей эксперимента считается тройка `method_id × seed × resolved configuration`. Для неё фиксируются версия Python, путь интерпретатора, версии RDKit/NumPy/Pandas/scikit-learn, платформа, команда запуска, абсолютные пути исходных файлов, SHA-256 каждого локального входа и полный registry внешних источников. URL без локально зафиксированного артефакта получают явный checksum-статус `NOT_FETCHED_NO_LOCAL_ARTIFACT`, а не фиктивный хеш. После завершения создаётся `artifact_manifest.json` с размером и SHA-256 каждого результата. Это позволяет отличить научно значимое изменение конфигурации от случайной подмены входного файла.

Исходные XLSX/DOCX/PPTX не копируются поверх и не редактируются. Таблицы `reviewer_training.csv` и `reaction_library.csv` являются производными и живут только внутри каталога запуска. Каждая обучающая строка несёт `source_id`, `source_kind`, `evidence_tier`, состояние фотопереключателя, температуру, среду и уровень метки. Такая схема нужна, чтобы молекулярную метку не спутать с измерением плёнки или готовой формуляции. В production-наборе рядом с нормализованным значением должны храниться исходное значение, исходная единица и правило преобразования; fixture-набор использует уже фиксированные единицы и не маскируется под эксперимент.

Scaffold split рассчитывается до fit. Один и тот же generic Murcko scaffold не может появиться в разных split, а loader повторно проверяет пересечение и останавливает обучение при leakage. Validation служит выбору/калибровке, test — финальной диагностике. ITI обозначен только как validation-only class shift и не включён в три генеративных семейства. Малые M03/M04 должны использоваться прежде всего для физической проверки и оценки переноса; литературные M08–M10 не трактуются как доступные виртуальные библиотеки. Регуляторные U01/U03/U06 остаются справочниками, а не автоматически размеченными обучающими строками.

## Приложение B. Как читать метрики и отрицательный результат

Validity отвечает только за машинно проверяемую химическую корректность и не равна устойчивости соединения. Uniqueness считается по каноническому SMILES внутри запуска. Novelty сравнивает результат с фактическим training set, но не с мировой химической литературой. Internal diversity — среднее расстояние Morgan fingerprints на детерминированной выборке пар; scaffold diversity — доля generic Murcko scaffold. SA-score в этом прототипе — прозрачный complexity proxy, не оценка маршрута синтеза. Поэтому высокая novelty или низкий SA не означают коммерческую доступность продукта реакции.

Joint success — наиболее строгая метрика: она требует одновременно UV-pass, MOST-pass, safety-pass и обе AD. Доверительные границы здесь важнее средних значений. Например, молекула с высоким средним UVA AUC отвергается, если ensemble расходится настолько, что нижняя граница не достигает семейного референса. Аналогично положительное среднее ΔH не помогает за пределами family-specific MOST domain. В phototoxicity применяется ещё более консервативное правило: неопределённая область сама по себе блокирует shortlist. Такой порядок уменьшает число эффектных, но неподтверждаемых виртуальных лидов.

Bootstrap выполняется по трём независимым seed-результатам, а не по 3000 молекулам как будто они независимые эксперименты. Поэтому интервал отражает вариабельность запуска метода, хотя при трёх seeds он остаётся ориентировочным. `metrics_runs.csv` сохраняет исходные значения, а `metrics_summary.csv` — среднее и percentile interval. Для сравнения методов нужно одновременно смотреть joint success и diversity: метод с немного большим success, но с collapse к одному кластеру, не обязательно предпочтительнее. `reward_diagnostics.csv` показывает effective sample size и максимальную долю одного ECFP-кластера; крайне малый ESS сигнализирует, что несколько молекул доминируют в обучающем сигнале.

Абляция uncertainty показывает, сколько кандидатов появилось бы при использовании только средних прогнозов. Абляция safety или AD не является альтернативным допустимым shortlist — это диагностическая карта конфликта требований. Абляция diversity сравнивает кластерное покрытие top-100 при ранжировании исходной и штрафованной наградой. Curriculum оценивается относительно двух matched-budget baseline, но причинный вывод о пользе curriculum требует повторения на реальных моделях и данных. Если joint-кандидатов нет, следует изучить распределения отдельных gate и failure reasons, а не ослаблять пороги постфактум.

## Приложение C. Переход от прототипа к физическому исследованию

`train-generator` создаёт отдельный REINVENT4 v{config['production']['reinvent4_version']} TOML для каждого семейства, scaffold-файл с двумя attachment points и ExternalProcess bridge к тем же reward-reviewers. Конфигурация использует staged learning, DAP, четыре curriculum stages и PenalizeSameSmiles. Суммарный batch/step budget приблизительно согласован с matched reviewer budget. Production preflight требует установленный `reinvent`, существующий LibInvent prior, совпадающий SHA-256 и полный xTB/sTDA toolchain. Отсутствие любого обязательного артефакта завершает запуск ошибкой до выдачи научного результата.

Для physical oracle RDKit создаёт восемь конформеров обоих состояний, выполняет MMFF/UFF pre-relaxation и сохраняет лучший XYZ. Затем GFN2-xTB должен независимо оптимизировать обе структуры; разность электронных энергий не подменяет свободную энергию, но служит внешней проверкой знака и масштаба storage proxy. sTDA-xTB даёт переходы, которые необходимо уширить с явно записанной шириной линии и повторно интегрировать по UVB/UVA. Расчёты не возвращаются во внутренний RL loop, поэтому сохраняется независимость внешней проверки.

Даже успешный physical oracle не делает молекулу безопасным ингредиентом. До формуляции нужны синтез и аналитическое подтверждение структуры, выделение обоих состояний, измерение спектра и quantum yield, проверка обратимости/усталости, t½ при нескольких температурах и оценка побочных фотопродуктов. Safety-пакет должен включать растворимость, проникновение через кожу, раздражение, сенсибилизацию и экспериментальную фототоксичность. Только после этого имеет смысл изготовить воспроизводимую плёнку с контролируемой загрузкой и сравнивать готовый продукт применимыми методами ISO. Молекулярный расчёт остаётся способом приоритизации, а не заменой этой цепочки доказательств.

Источники: [REINVENT4](https://github.com/MolecularAI/REINVENT4), [SCCP opinion](https://ec.europa.eu/health/ph_risk/committees/04_sccp/docs/sccp_o_036.pdf), [ISO 24444](https://www.iso.org/standard/72250.html), [ISO 24443](https://www.iso.org/standard/75059.html), [OECD TG 432](https://www.oecd.org/en/publications/2019/06/test-no-432-in-vitro-3t3-nru-phototoxicity-test_g1gh4b69.html).
"""
    report_path = output / "report.md"
    report_path.write_text(report, encoding="utf-8")
    slides = f"""# 7-минутная презентация

## Слайд 1 — Задача и честная граница утверждений (0:00–0:45)

- Одна UV-поглощающая MOST-молекула, не готовая формуляция.
- Пересечение UVB/UVA, энергии, 4–24 ч и safety triage.
- Только screening proxy; не ISO-сертификация и не доказательство безопасности.

## Слайд 2 — Три family-aware пространства (0:45–1:35)

- NBD/QC: основной класс; Dewar-pyrimidinone: low-data; spiropyran: transferability.
- Reaction-constrained синтоны и фиксированные позиции.
- RDKit: валентность, формула изомерной пары, canonical identity.

## Слайд 3 — Matched-budget дизайн (1:35–2:25)

- Prior random vs weighted retraining vs LibInvent curriculum surrogate.
- {config['execution']['reviewer_budget_per_run']} reviewer-вызовов на method × seed.
- REINVENT4 manifests отделены от CPU smoke backend.

## Слайд 4 — Reward и reviewer independence (2:25–3:25)

- Spectrum 290–400 нм → UVB/UVA AUC, λc, Beer–Lambert proxy.
- Family-specific ΔH/Wh·kg⁻¹/log(t½); safety Kp/phototoxicity.
- Random Forest reward ≠ Extra Trees evaluator; scaffold split + AD + uncertainty.

## Слайд 5 — Жёсткая безопасность (3:25–4:20)

- Veto псораленов/фурокумаринов до reward.
- Known phototoxic/reactive/unsupported/invalid pair — немедленный отказ.
- Uncertain phototoxicity никогда не проходит в shortlist.

## Слайд 6 — Результаты (4:20–5:40)

{chr(10).join(table)}

- Generated: {len(generated)}; independent reviewed: {review['reviewed']}.
- Provisional: {review['independent_joint_pass']}; final after oracle: {review['selected_after_physical_oracle']}.
- Oracle status: `{review['physical_oracle']['status']}`.

## Слайд 7 — Решение и следующий шаг (5:40–7:00)

- Отсутствие joint-pass — информативный результат, не доказательство невозможности.
- Заменить fixture labels реальными M/U datasets; pin prior + commit + checksums.
- GFN2/sTDA-xTB top-50–100 → синтез → OECD TG 432 → испытание плёнки/формуляции.
"""
    slides_path = output / "presentation_7min.md"
    slides_path.write_text(slides, encoding="utf-8")
    return {"report": str(report_path.resolve()), "presentation": str(slides_path.resolve())}
'''
_load_embedded_module('mostgen.reporting', _source, PROJECT_ROOT / 'mostgen' / 'reporting.py')
print('loaded mostgen.reporting')


loaded mostgen.reporting


In [17]:
_source = r'''from __future__ import annotations

import json
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

from .data import read_csv


def _truth(value: Any) -> bool:
    return value is True or str(value).lower() in {"true", "1", "yes"}


def verify_experiment(root: str | Path, config: dict[str, Any]) -> dict[str, Any]:
    experiment = Path(root)
    rows = read_csv(experiment / "generated.csv")
    grouped: dict[tuple[str, str], list[dict[str, str]]] = defaultdict(list)
    for row in rows:
        grouped[(row["method_id"], row["seed"])].append(row)
    expected_runs = {
        (method, str(seed))
        for method in config["execution"]["methods"]
        for seed in config["execution"]["seeds"]
    }
    minimum = int(config["execution"]["n_per_run"])
    required_columns = {
        "smiles", "charged_smiles", "family", "method_id", "seed", "uvb_auc", "uva_auc",
        "lambda_c_nm", "energy_kj_mol", "specific_energy_wh_kg", "half_life_h",
        "phototoxicity_probability", "phototoxicity_uncertainty", "similarity_D_A", "similarity_D_B",
        "ad_spectral", "ad_most", "sa_score", "psoralen_alert", "joint_pass", "selected",
        "not_iso_certified",
    }
    checks = {
        "all_expected_runs_present": set(grouped) == expected_runs,
        "minimum_unique_per_run": all(len({row["smiles"] for row in members}) >= minimum for members in grouped.values()),
        "exact_matched_reviewer_budget": all(len(members) == int(config["execution"]["reviewer_budget_per_run"]) for members in grouped.values()),
        "all_three_families": set(row["family"] for row in rows) == set(config["families"]),
        "zero_psoralen_cores": not any(_truth(row.get("psoralen_alert")) for row in rows),
        "zero_known_phototoxic_matches": not any(_truth(row.get("known_phototoxic_match")) for row in rows),
        "no_uncertain_phototoxicity_selected": not any(_truth(row.get("selected")) and _truth(row.get("phototoxicity_uncertain")) for row in rows),
        "no_iso_claims": all(_truth(row.get("not_iso_certified")) for row in rows),
        "required_generated_schema": bool(rows) and required_columns <= set(rows[0]),
        "reward_evaluator_separation": (experiment / "models" / "reward" / "reviewers.pkl").resolve() != (experiment / "models" / "evaluator" / "reviewers.pkl").resolve(),
        "metrics_present": all((experiment / "metrics" / name).is_file() for name in ("metrics_runs.csv", "metrics_summary.csv", "ablations.csv", "reward_diagnostics.csv")),
        "report_present": (experiment / "report" / "report.md").is_file(),
        "presentation_present": (experiment / "report" / "presentation_7min.md").is_file(),
        "gpu_budget_respected": float(config["execution"]["gpu_hours"]) <= float(config["project"]["max_gpu_hours"]),
    }
    verification = {
        "schema_version": "1.0",
        "passed": all(checks.values()),
        "checks": checks,
        "generated_rows": len(rows),
        "run_counts": {f"{method}:{seed}": len(members) for (method, seed), members in sorted(grouped.items())},
        "run_unique_counts": {f"{method}:{seed}": len({row["smiles"] for row in members}) for (method, seed), members in sorted(grouped.items())},
        "family_counts": dict(sorted(Counter(row["family"] for row in rows).items())),
        "scientific_exceptions": {
            "no_joint_candidates_is_allowed": True,
            "physical_oracle_unavailable_blocks_final_selection": True,
        },
    }
    (experiment / "verification.json").write_text(json.dumps(verification, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    if not verification["passed"]:
        failed = [name for name, passed in checks.items() if not passed]
        raise RuntimeError(f"Experiment acceptance verification failed: {failed}")
    return verification
'''
_load_embedded_module('mostgen.validation', _source, PROJECT_ROOT / 'mostgen' / 'validation.py')
print('loaded mostgen.validation')


loaded mostgen.validation


In [18]:
_source = r'''from __future__ import annotations

import argparse
import json
import shutil
import sys
from pathlib import Path
from typing import Any

from .config import apply_override, dump_resolved_config, load_config
from .data import prepare_data
from .metrics import compute_metrics
from .oracle import availability
from .provenance import append_event, sha256_file, write_artifact_manifest, write_manifest
from .reporting import build_reports
from .review import review_generated
from .reviewers import train_reviewers
from .search import run_methods, write_generator_manifests
from .validation import verify_experiment


ROOT = Path(__file__).resolve().parents[1]


def _config(args: argparse.Namespace) -> dict[str, Any]:
    return apply_override(load_config(args.config, args.mode), args.override)


def _layout(output: str | Path) -> dict[str, Path]:
    root = Path(output).resolve()
    return {
        "root": root, "data": root / "data", "models": root / "models",
        "generator": root / "generator", "generated": root / "generated.csv",
        "metrics": root / "metrics", "review": root / "review",
    }


def _initialize(config: dict[str, Any], paths: dict[str, Path]) -> None:
    paths["root"].mkdir(parents=True, exist_ok=True)
    dump_resolved_config(config, paths["root"] / "config.resolved.json")
    inputs = [config["_config_path"], ROOT / "pyproject.toml", ROOT / "requirements.txt"]
    inputs.extend(sorted((ROOT / "mostgen").glob("*.py")))
    inputs.append(ROOT / "scripts" / "reinvent_external_score.py")
    inputs.extend(ROOT / name for name in ("database_matrix_MOST_UV_skin.xlsx", "Задание.docx", "Солнцезащитная плёнка с молекулярным накоплением солнечной энергии.pptx"))
    write_manifest(paths["root"] / "experiment_manifest.json", config, inputs)


def _require(path: Path, hint: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}; run `{hint}` first")


def _production_checks(config: dict[str, Any]) -> None:
    production = config["production"]
    errors = []
    if production["reinvent4_revision"].startswith("PIN_REQUIRED"):
        errors.append("REINVENT4 revision is not pinned")
    prior = Path(production["reaction_prior_path"]) if production["reaction_prior_path"] else None
    if prior is None or not prior.is_file():
        errors.append("reaction prior path is not a file")
    elif not production.get("reaction_prior_sha256"):
        errors.append("reaction prior SHA-256 is not configured")
    elif sha256_file(prior) != production["reaction_prior_sha256"]:
        errors.append("reaction prior SHA-256 mismatch")
    if shutil.which("reinvent") is None:
        errors.append("REINVENT4 `reinvent` executable is unavailable")
    if production.get("require_physical_oracle") and not availability(config)["ready"]:
        errors.append("xTB/sTDA toolchain is unavailable")
    if errors:
        raise RuntimeError("Production preflight failed: " + "; ".join(errors))


def command_prepare(args: argparse.Namespace) -> dict[str, Any]:
    config, paths = _config(args), _layout(args.output)
    _initialize(config, paths)
    result = prepare_data(config, paths["data"], ROOT)
    append_event(paths["root"] / "events.jsonl", "prepare-data", result)
    return result


def command_train_reviewers(args: argparse.Namespace) -> dict[str, Any]:
    config, paths = _config(args), _layout(args.output)
    _initialize(config, paths)
    _require(paths["data"] / "reviewer_training.csv", "prepare-data")
    result = train_reviewers(config, paths["data"] / "reviewer_training.csv", paths["models"])
    append_event(paths["root"] / "events.jsonl", "train-reviewers", {"rows": result["rows"]})
    return result


def command_train_generator(args: argparse.Namespace) -> dict[str, Any]:
    config, paths = _config(args), _layout(args.output)
    _initialize(config, paths)
    if config["execution"]["mode"] == "production":
        _production_checks(config)
    result = write_generator_manifests(config, paths["generator"])
    append_event(paths["root"] / "events.jsonl", "train-generator", result)
    return result


def command_search(args: argparse.Namespace, methods: list[str]) -> dict[str, Any]:
    config, paths = _config(args), _layout(args.output)
    _initialize(config, paths)
    _require(paths["data"] / "reaction_library.csv", "prepare-data")
    _require(paths["models"] / "reward" / "reviewers.pkl", "train-reviewers")
    output_name = "generated.csv" if args.command in {"generate", "run-all"} else "baselines.csv"
    result = run_methods(
        config, paths["data"] / "reaction_library.csv", paths["models"] / "reward" / "reviewers.pkl",
        paths["root"] / output_name, methods,
    )
    append_event(paths["root"] / "events.jsonl", args.command, result)
    return result


def command_review(args: argparse.Namespace) -> dict[str, Any]:
    config, paths = _config(args), _layout(args.output)
    _initialize(config, paths)
    _require(paths["generated"], "generate")
    _require(paths["models"] / "evaluator" / "reviewers.pkl", "train-reviewers")
    result = review_generated(config, paths["generated"], paths["models"] / "evaluator" / "reviewers.pkl", paths["review"])
    append_event(paths["root"] / "events.jsonl", "review", {"reviewed": result["reviewed"], "selected": result["selected_after_physical_oracle"]})
    return result


def command_run_all(args: argparse.Namespace) -> dict[str, Any]:
    config, paths = _config(args), _layout(args.output)
    _initialize(config, paths)
    data_result = prepare_data(config, paths["data"], ROOT)
    append_event(paths["root"] / "events.jsonl", "prepare-data", data_result)
    model_result = train_reviewers(config, paths["data"] / "reviewer_training.csv", paths["models"])
    append_event(paths["root"] / "events.jsonl", "train-reviewers", {"rows": model_result["rows"]})
    generator_result = write_generator_manifests(config, paths["generator"])
    if config["execution"]["mode"] == "production":
        _production_checks(config)
    search_result = run_methods(
        config, paths["data"] / "reaction_library.csv", paths["models"] / "reward" / "reviewers.pkl",
        paths["generated"], list(config["execution"]["methods"]),
    )
    metrics_result = compute_metrics(paths["generated"], paths["data"] / "reviewer_training.csv", paths["metrics"], config)
    review_result = review_generated(config, paths["generated"], paths["models"] / "evaluator" / "reviewers.pkl", paths["review"])
    reports = build_reports(paths["root"], config)
    verification = verify_experiment(paths["root"], config)
    result = {
        "experiment": str(paths["root"]), "data_rows": data_result["training_rows"],
        "generated_rows": search_result["rows"], "run_counts": search_result["run_counts"],
        "matched_budget": metrics_result["matched_reviewer_budget"],
        "independent_reviewed": review_result["reviewed"],
        "selected": review_result["selected_after_physical_oracle"], "reports": reports,
        "generator_status": generator_result["status"],
        "acceptance_verified": verification["passed"],
    }
    append_event(paths["root"] / "events.jsonl", "run-all-complete", result)
    (paths["root"] / "run_summary.json").write_text(json.dumps(result, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    write_artifact_manifest(paths["root"])
    return result


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(prog="mostgen", description="Reproducible UV/MOST screening pipeline")
    subparsers = parser.add_subparsers(dest="command", required=True)
    help_text = {
        "prepare-data": "prepare immutable-source derivatives and scaffold splits",
        "train-reviewers": "train separate reward and evaluator ensembles",
        "sample-baselines": "run matched-budget prior and weighted baselines",
        "train-generator": "write family-specific REINVENT4/LibInvent manifests",
        "generate": "run curriculum LibInvent surrogate generation",
        "review": "independently review top candidates and queue physical oracle",
        "run-all": "execute the entire reproducible workflow",
    }
    for name, description in help_text.items():
        child = subparsers.add_parser(name, help=description)
        child.add_argument("--config", default=str(ROOT / "config" / "default.json"))
        child.add_argument("--override", help="JSON configuration patch")
        child.add_argument("--mode", choices=("smoke", "full", "production"), default="full")
        child.add_argument("--output", default="runs/latest")
    return parser


def main(argv: list[str] | None = None) -> int:
    parser = build_parser()
    args = parser.parse_args(argv)
    try:
        if args.command == "prepare-data":
            result = command_prepare(args)
        elif args.command == "train-reviewers":
            result = command_train_reviewers(args)
        elif args.command == "sample-baselines":
            result = command_search(args, ["prior_random", "weighted_retraining"])
        elif args.command == "train-generator":
            result = command_train_generator(args)
        elif args.command == "generate":
            result = command_search(args, ["libinvent_rl"])
        elif args.command == "review":
            result = command_review(args)
        else:
            result = command_run_all(args)
        print(json.dumps(result, indent=2, sort_keys=True, default=str))
        return 0
    except Exception as exc:
        print(f"ERROR: {exc}", file=sys.stderr)
        return 2
'''
_load_embedded_module('mostgen.cli', _source, PROJECT_ROOT / 'mostgen' / 'cli.py')
print('loaded mostgen.cli')


loaded mostgen.cli


In [19]:
_source = r'''from .cli import main


if __name__ == "__main__":
    raise SystemExit(main())

'''
_load_embedded_module('mostgen.__main__', _source, PROJECT_ROOT / 'mostgen' / '__main__.py')
print('loaded mostgen.__main__')


loaded mostgen.__main__


In [20]:
REINVENT_EXTERNAL_SCORER_SOURCE = r'''#!/usr/bin/env python3
"""REINVENT4 ExternalProcess bridge for MOSTGen reward reviewers.

REINVENT passes one full SMILES per stdin line.  The bridge deliberately scores
only canonical structures present in the versioned reaction library, thereby
keeping neural LibInvent generation inside the same commercial-synthon space
as both baselines. Unknown, invalid, vetoed, or wrong-family structures receive
zero. Output follows REINVENT4 ExternalProcess payload version 1.
"""
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

from mostgen.chemistry import ChemistryError, standardize_smiles  # noqa: E402
from mostgen.config import load_config  # noqa: E402
from mostgen.data import read_csv  # noqa: E402
from mostgen.reviewers import ReviewerBundle  # noqa: E402
from mostgen.scoring import ScoringContext  # noqa: E402


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    parser.add_argument("--model", required=True)
    parser.add_argument("--library", required=True)
    parser.add_argument("--family", required=True)
    parser.add_argument("--stage", choices=("chemistry", "spectrum", "most", "safety"), required=True)
    return parser


def main(argv: list[str] | None = None) -> int:
    args = build_parser().parse_args(argv)
    config = load_config(args.config, mode="full")
    library = {}
    for row in read_csv(args.library):
        if row["family"] == args.family:
            library[row["smiles"]] = row
    requested = [line.strip() for line in sys.stdin if line.strip()]
    candidates = []
    positions = []
    scores = [0.0] * len(requested)
    for index, smiles in enumerate(requested):
        try:
            canonical = standardize_smiles(smiles)
        except ChemistryError:
            continue
        candidate = library.get(canonical)
        if candidate is not None:
            candidates.append(candidate)
            positions.append(index)
    if candidates:
        context = ScoringContext(config, ReviewerBundle.load(args.model))
        reviewed = context.review_candidates(candidates, "reinvent4_external", int(config["project"]["default_seed"]))
        for position, row in zip(positions, reviewed):
            if row.get("safety_veto") or not row.get("valid"):
                value = 0.0
            else:
                stages = json.loads(row.get("stage_rewards_json", "{}"))
                value = float(stages.get(args.stage, 0.0))
                if args.stage == "safety" and row.get("phototoxicity_uncertain"):
                    value = 0.0
            scores[position] = max(0.0, min(1.0, value))
    print(json.dumps({"version": 1, "payload": {"stage_score": scores}}))
    return 0



if __name__ == "__main__":
    raise SystemExit(main())
'''
print('embedded external scorer lines:', len(REINVENT_EXTERNAL_SCORER_SOURCE.splitlines()))


embedded external scorer lines: 75


In [21]:
CONFIG_JSON = r'''{
  "schema_version": "1.0",
  "project": {
    "name": "mostgen_uv_most",
    "version": "0.1.0",
    "default_seed": 1701,
    "max_training_structures": 30000,
    "max_gpu_hours": 8.0,
    "disclaimer": "Research screening proxy; not ISO-certified and not a cosmetic safety determination."
  },
  "execution": {
    "backend": "surrogate_smoke",
    "methods": ["prior_random", "weighted_retraining", "libinvent_rl"],
    "seeds": [1701, 2903, 4211],
    "n_per_run": 1000,
    "reviewer_budget_per_run": 1000,
    "shortlist_size": 75,
    "bootstrap_samples": 300,
    "gpu_hours": 0.0
  },
  "smoke": {
    "seeds": [17],
    "n_per_run": 40,
    "reviewer_budget_per_run": 40,
    "shortlist_size": 12,
    "bootstrap_samples": 40,
    "training_rows_per_family": 72
  },
  "production": {
    "reinvent4_repository": "https://github.com/MolecularAI/REINVENT4",
    "reinvent4_version": "4.8",
    "reinvent4_revision": "80a8d21",
    "reaction_prior_path": "",
    "reaction_prior_sha256": "",
    "xtb_executable": "xtb",
    "stda_executable": "stda",
    "xtb4stda_executable": "xtb4stda",
    "require_rdkit": true,
    "require_physical_oracle": true
  },
  "elements": ["B", "C", "N", "O", "F", "Si", "P", "S", "Cl", "Br"],
  "validation_only_families": ["iti"],
  "families": {
    "nbd_qc": {
      "role": "primary",
      "scaffold_id": "NBD-1,4-disubstituted",
      "ground_template": "C1({r1})=CC2C=CC1({r2})C2",
      "charged_template": "C1({r1})C2C3C4C1({r2})C3C24",
      "reaction_smarts": "[*:1].[C:2]1=[C:3][C:4]2[C:5]=[C:6][C:7]1[C:8]2>>[*:1]-[C:2]1=[C:3][C:4]2[C:5]=[C:6][C:7]1[C:8]2",
      "allowed_positions": ["C1", "C4"]
    },
    "dewar_pyrimidinone": {
      "role": "low_data_experimental",
      "scaffold_id": "pyrimidin-4-one-2,6-disubstituted",
      "ground_template": "O=c1[nH]c({r1})nc({r2})c1",
      "charged_template": "O=C1N2C({r1})=NC({r2})C12",
      "reaction_smarts": "[*:1].[O:2]=[c:3]1[nH:4][c:5][n:6][c:7][c:8]1>>[O:2]=[c:3]1[nH:4][c:5]([*:1])[n:6][c:7][c:8]1",
      "allowed_positions": ["C2", "C6"]
    },
    "spiropyran": {
      "role": "transferability_check",
      "scaffold_id": "indolino-benzopyran-6,8-disubstituted",
      "ground_template": "c1c({r1})cc2c(c1{r2})OC1(CCN(C)C1)C2",
      "charged_template": "C[NH+]1CCC(C1)=Cc1c([O-])cc({r1})cc1{r2}",
      "reaction_smarts": "[*:1].[c:2]1[c:3][c:4][c:5]2[c:6]([c:7]1)[O:8][C:9]1(CC[N:10](C)C1)[C:11]2>>[*:1]-[c:2]1[c:3][c:4][c:5]2[c:6]([c:7]1)[O:8][C:9]1(CC[N:10](C)C1)[C:11]2",
      "allowed_positions": ["C6", "C8"]
    }
  },
  "synthons": [
    {"id": "S01", "smiles": "F", "commercial": true},
    {"id": "S02", "smiles": "Cl", "commercial": true},
    {"id": "S03", "smiles": "Br", "commercial": true},
    {"id": "S04", "smiles": "C", "commercial": true},
    {"id": "S05", "smiles": "CC", "commercial": true},
    {"id": "S06", "smiles": "CCC", "commercial": true},
    {"id": "S07", "smiles": "C(F)(F)F", "commercial": true},
    {"id": "S08", "smiles": "C#N", "commercial": true},
    {"id": "S09", "smiles": "N", "commercial": true},
    {"id": "S10", "smiles": "NC", "commercial": true},
    {"id": "S11", "smiles": "N(C)C", "commercial": true},
    {"id": "S12", "smiles": "O", "commercial": true},
    {"id": "S13", "smiles": "OC", "commercial": true},
    {"id": "S14", "smiles": "OCC", "commercial": true},
    {"id": "S15", "smiles": "C(=O)O", "commercial": true},
    {"id": "S16", "smiles": "C(=O)OC", "commercial": true},
    {"id": "S17", "smiles": "C(=O)N", "commercial": true},
    {"id": "S18", "smiles": "S(=O)(=O)C", "commercial": true},
    {"id": "S19", "smiles": "c5ccccc5", "commercial": true},
    {"id": "S20", "smiles": "c5ccc(F)cc5", "commercial": true},
    {"id": "S21", "smiles": "c5ccc(Cl)cc5", "commercial": true},
    {"id": "S22", "smiles": "c5ccc(C#N)cc5", "commercial": true},
    {"id": "S23", "smiles": "c5ccc(OC)cc5", "commercial": true},
    {"id": "S24", "smiles": "c5ccc(N(C)C)cc5", "commercial": true},
    {"id": "S25", "smiles": "c5ccncc5", "commercial": true},
    {"id": "S26", "smiles": "c5ncccc5", "commercial": true},
    {"id": "S27", "smiles": "c5ccsc5", "commercial": true},
    {"id": "S28", "smiles": "C=C", "commercial": true},
    {"id": "S29", "smiles": "C=C(C)C", "commercial": true},
    {"id": "S30", "smiles": "C5CC5", "commercial": true},
    {"id": "S31", "smiles": "C5CCC5", "commercial": true},
    {"id": "S32", "smiles": "C5CCOC5", "commercial": true},
    {"id": "S33", "smiles": "C(F)F", "commercial": true},
    {"id": "S34", "smiles": "P(=O)(O)O", "commercial": true}
  ],
  "reviewers": {
    "ensemble_size": 5,
    "fingerprint_bits": 256,
    "ad_similarity_threshold": 0.18,
    "confidence_z": 1.645,
    "half_life_window_hours": [4.0, 24.0],
    "lambda_c_min_nm": 370.0,
    "phototoxicity_max_probability": 0.35,
    "kp_max_log_cm_s": -4.0,
    "uncertain_probability_band": [0.25, 0.45]
  },
  "reward": {
    "floor": 0.001,
    "stages": [
      {"name": "chemistry", "components": ["validity", "family", "pair"]},
      {"name": "spectrum", "components": ["uvb", "uva", "lambda_c"]},
      {"name": "most", "components": ["energy", "half_life"]},
      {"name": "safety", "components": ["phototoxicity", "permeation", "sa", "ad"]}
    ],
    "weights": {"uvb": 1.2, "uva": 1.5, "lambda_c": 1.0, "energy": 1.2, "half_life": 1.0, "phototoxicity": 1.5, "permeation": 0.8, "sa": 0.6, "ad": 1.0},
    "diversity_penalty": 0.12
  },
  "safety": {
    "psoralen_similarity_warning_threshold": 0.45,
    "psoralen_core_smarts": [
      "C1=CC(=O)OC2=CC3=C(C=CO3)C=C21",
      "C1=CC2=C(C=CO2)C3=C1C=CC(=O)O3"
    ],
    "known_phototoxic_smiles": [
      "C1=CC(=O)OC2=CC3=C(C=CO3)C=C21",
      "C1=CC2=C(C=CO2)C3=C1C=CC(=O)O3",
      "COC1=C2C=CC(=O)OC2=CC3=C1C=CO3"
    ],
    "reactive_fragments": ["N=N=N", "OO", "C(Cl)(Cl)Cl", "[N-]=[N+]=N", "P(Cl)Cl", "S(Cl)(=O)=O"]
  },
  "sources": [
    {"id": "M01", "role": "photoswitch calibration", "url": "LOCAL_OR_LICENSED_DATA_REQUIRED", "license": "verify_before_use"},
    {"id": "M03", "role": "MOST energy and spectrum validation", "url": "LOCAL_OR_LICENSED_DATA_REQUIRED", "license": "verify_before_use"},
    {"id": "M04", "role": "MOST state validation", "url": "LOCAL_OR_LICENSED_DATA_REQUIRED", "license": "verify_before_use"},
    {"id": "M08-M10", "role": "literature orientation only; no bulk training data claimed", "url": "BIBLIOGRAPHY_RESOLUTION_REQUIRED", "license": "metadata_only"},
    {"id": "M11", "role": "spectral pretraining", "url": "LOCAL_DATA_REQUIRED", "license": "verify_before_use"},
    {"id": "M12", "role": "spectral pretraining", "url": "LOCAL_DATA_REQUIRED", "license": "verify_before_use"},
    {"id": "M13", "role": "unified spectral transfer", "url": "LOCAL_DATA_REQUIRED", "license": "verify_before_use"},
    {"id": "U07-U10", "role": "irritation and sensitisation", "url": "LOCAL_DATA_REQUIRED", "license": "verify_before_use"},
    {"id": "U11-U14", "role": "skin permeation Kp/Jmax", "url": "LOCAL_DATA_REQUIRED", "license": "verify_before_use"},
    {"id": "U16-U17", "role": "phototoxicity", "url": "LOCAL_DATA_REQUIRED", "license": "verify_before_use"},
    {"id": "U01/U03/U06", "role": "versioned regulatory and expert references only", "url": "REGISTRY_RESOLUTION_REQUIRED", "license": "metadata_only"},
    {"id": "REINVENT4", "role": "LibInvent generator", "url": "https://github.com/MolecularAI/REINVENT4", "version": "4.8@80a8d21", "license": "Apache-2.0"},
    {"id": "SCCP", "role": "furocoumarin safety opinion", "url": "https://ec.europa.eu/health/ph_risk/committees/04_sccp/docs/sccp_o_036.pdf", "license": "public_document"},
    {"id": "ISO24444", "role": "finished-product in vivo SPF method", "url": "https://www.iso.org/standard/72250.html", "license": "metadata_only"},
    {"id": "ISO24443", "role": "finished-product in vitro UVA method", "url": "https://www.iso.org/standard/75059.html", "license": "metadata_only"},
    {"id": "OECD432", "role": "experimental phototoxicity", "url": "https://www.oecd.org/en/publications/2019/06/test-no-432-in-vitro-3t3-nru-phototoxicity-test_g1gh4b69.html", "license": "public_metadata"}
  ]
}
'''
from mostgen.config import validate_config
RUN_MODE = 'smoke'  # замените на 'full' для 3 × 3 × 1000
config = json.loads(CONFIG_JSON)
config['_config_path'] = str(PROJECT_ROOT / 'config' / 'default.json')
if RUN_MODE == 'smoke':
    smoke = dict(config['smoke'])
    config['execution'].update({k: v for k, v in smoke.items() if k != 'training_rows_per_family'})
    config['execution']['mode'] = 'smoke'
    config['training_rows_per_family'] = smoke['training_rows_per_family']
else:
    config['execution']['mode'] = 'full'
    config['training_rows_per_family'] = 180
validate_config(config)
display({k: config['execution'][k] for k in ['mode', 'methods', 'seeds', 'n_per_run', 'reviewer_budget_per_run', 'shortlist_size']})


{'mode': 'smoke',
 'methods': ['prior_random', 'weighted_retraining', 'libinvent_rl'],
 'seeds': [17],
 'n_per_run': 40,
 'reviewer_budget_per_run': 40,
 'shortlist_size': 12}

In [22]:
from mostgen.config import dump_resolved_config
from mostgen.provenance import append_event, write_manifest

OUTPUT_DIR = PROJECT_ROOT / "runs" / f"notebook-{RUN_MODE}"
DATA_DIR = OUTPUT_DIR / "data"
MODELS_DIR = OUTPUT_DIR / "models"
GENERATOR_DIR = OUTPUT_DIR / "generator"
METRICS_DIR = OUTPUT_DIR / "metrics"
REVIEW_DIR = OUTPUT_DIR / "review"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
dump_resolved_config(config, OUTPUT_DIR / "config.resolved.json")
source_inputs = [
    PROJECT_ROOT / "GOSHA.ipynb", PROJECT_ROOT / "config" / "default.json",
    PROJECT_ROOT / "database_matrix_MOST_UV_skin.xlsx",
    PROJECT_ROOT / "Задание.docx",
    PROJECT_ROOT / "Солнцезащитная плёнка с молекулярным накоплением солнечной энергии.pptx",
]
write_manifest(OUTPUT_DIR / "experiment_manifest.json", config, source_inputs, ["GOSHA.ipynb", "Run All"])
print("output:", OUTPUT_DIR)


output: /home/sigmatau17/TEST/runs/notebook-smoke


In [23]:
from mostgen.data import prepare_data, read_csv

data_result = prepare_data(config, DATA_DIR, PROJECT_ROOT)
append_event(OUTPUT_DIR / "events.jsonl", "prepare-data", data_result)
display({k: data_result[k] for k in ["training_rows", "library_rows", "raw_inputs_mutated", "warning"]})


{'training_rows': 216,
 'library_rows': 3468,
 'raw_inputs_mutated': False,
 'warning': 'Fixture labels are synthetic and cannot support scientific, efficacy, or safety claims.'}

In [24]:
from mostgen.reviewers import train_reviewers

model_result = train_reviewers(config, DATA_DIR / "reviewer_training.csv", MODELS_DIR)
append_event(OUTPUT_DIR / "events.jsonl", "train-reviewers", {"rows": model_result["rows"]})
display({
    "rows": model_result["rows"],
    "evidence_tiers": model_result["evidence_tiers"],
    "independent_instances": model_result["independent_instances"],
    "reward_algorithm": model_result["reward"]["algorithm"],
    "evaluator_algorithm": model_result["evaluator"]["algorithm"],
})


{'rows': 216,
 'evidence_tiers': ['synthetic_smoke_only'],
 'independent_instances': True,
 'reward_algorithm': 'RandomForestRegressor ensemble',
 'evaluator_algorithm': 'ExtraTreesRegressor ensemble'}

In [25]:
from mostgen.search import write_generator_manifests

generator_result = write_generator_manifests(config, GENERATOR_DIR)
append_event(OUTPUT_DIR / "events.jsonl", "train-generator", generator_result)
display(generator_result)


{'status': 'manifest_only',
 'family_manifests': ['/home/sigmatau17/TEST/runs/notebook-smoke/generator/reinvent4_nbd_qc.json',
  '/home/sigmatau17/TEST/runs/notebook-smoke/generator/reinvent4_dewar_pyrimidinone.json',
  '/home/sigmatau17/TEST/runs/notebook-smoke/generator/reinvent4_spiropyran.json'],
 'reinvent_toml_configs': ['/home/sigmatau17/TEST/runs/notebook-smoke/generator/reinvent4_nbd_qc.toml',
  '/home/sigmatau17/TEST/runs/notebook-smoke/generator/reinvent4_dewar_pyrimidinone.toml',
  '/home/sigmatau17/TEST/runs/notebook-smoke/generator/reinvent4_spiropyran.toml'],
 'reinvent_command': 'reinvent -l <family>.log <family>.toml',
 'note': 'The smoke policy is an adaptive reaction-library search, not a trained REINVENT neural prior.'}

In [26]:
from mostgen.search import run_methods

generated_path = OUTPUT_DIR / "generated.csv"
search_result = run_methods(
    config,
    DATA_DIR / "reaction_library.csv",
    MODELS_DIR / "reward" / "reviewers.pkl",
    generated_path,
    list(config["execution"]["methods"]),
)
append_event(OUTPUT_DIR / "events.jsonl", "generate", search_result)
display(search_result)


{'path': '/home/sigmatau17/TEST/runs/notebook-smoke/generated.csv',
 'rows': 120,
 'run_counts': {'prior_random:17': 40,
  'weighted_retraining:17': 40,
  'libinvent_rl:17': 40},
 'methods': ['prior_random', 'weighted_retraining', 'libinvent_rl'],
 'seeds': [17]}

In [27]:
import pandas as pd
from mostgen.metrics import compute_metrics

metrics_result = compute_metrics(generated_path, DATA_DIR / "reviewer_training.csv", METRICS_DIR, config)
display(pd.read_csv(METRICS_DIR / "metrics_summary.csv"))
display(pd.read_csv(METRICS_DIR / "reward_diagnostics.csv"))


,method_id,seed_runs,validity_mean,validity_ci_low,validity_ci_high,uniqueness_mean,uniqueness_ci_low,uniqueness_ci_high,novelty_mean,novelty_ci_low,...,mean_sa_score_ci_high,joint_success_mean,joint_success_ci_low,joint_success_ci_high,both_ad_fraction_mean,both_ad_fraction_ci_low,both_ad_fraction_ci_high,family_coverage_mean,family_coverage_ci_low,family_coverage_ci_high
0,libinvent_rl,1,1.0,1.0,1.0,1.0,1.0,1.0,0.900,0.900,...,2.138279,0.025,0.025,0.025,1.0,1.0,1.0,1.0,1.0,1.0
1,prior_random,1,1.0,1.0,1.0,1.0,1.0,1.0,0.975,0.975,...,2.187304,0.000,0.000,0.000,1.0,1.0,1.0,1.0,1.0,1.0
2,weighted_retraining,1,1.0,1.0,1.0,1.0,1.0,1.0,0.875,0.875,...,2.160760,0.050,0.050,0.050,1.0,1.0,1.0,1.0,1.0,1.0


,method_id,seed,nonzero_fraction,effective_sample_size,largest_cluster_fraction,component_ad_mean,component_energy_mean,component_half_life_mean,component_lambda_c_mean,component_permeation_mean,component_phototoxicity_mean,component_sa_mean,component_uva_mean,component_uvb_mean
0,libinvent_rl,17,1.0,39.367820,0.075,1.0,0.448527,0.864920,0.940398,0.999983,0.681358,0.972916,0.458622,0.465554
1,prior_random,17,1.0,39.197354,0.100,1.0,0.429534,0.864202,0.938948,0.999988,0.649817,0.972735,0.449742,0.470247
2,weighted_retraining,17,1.0,39.558294,0.075,1.0,0.465442,0.860204,0.937964,0.999971,0.684766,0.972151,0.428326,0.454508


In [28]:
from mostgen.review import review_generated

review_result = review_generated(
    config, generated_path, MODELS_DIR / "evaluator" / "reviewers.pkl", REVIEW_DIR
)
append_event(OUTPUT_DIR / "events.jsonl", "review", {
    "reviewed": review_result["reviewed"],
    "selected": review_result["selected_after_physical_oracle"],
})
display(review_result)


{'reviewed': 12,
 'independent_joint_pass': 1,
 'selected_after_physical_oracle': 0,
 'physical_oracle': {'schema_version': '1.0',
  'availability': {'tools': {'xtb': None, 'stda': None, 'xtb4stda': None},
   'ready': False},
  'queued': 8,
  'errors': [{'candidate_id': '5ddf3ef4c5a6a01fb032',
    'error': 'RDKit conformer embedding failed'},
   {'candidate_id': 'a27da246ca00ffc4d81c',
    'error': 'RDKit conformer embedding failed'},
   {'candidate_id': 'c8515233311186e4947e',
    'error': 'RDKit conformer embedding failed'},
   {'candidate_id': '3076ea469d41f5b4fd02',
    'error': 'RDKit conformer embedding failed'}],
  'calculation_contract': {'energy': 'independent GFN2-xTB optimization and ground/charged energy difference',
   'spectrum': 'sTDA-xTB transitions broadened over 290-400 nm',
   'inside_rl_loop': False,
   'automatic_proxy_substitution': False},
  'status': 'not_run_external_tools_unavailable'},
 'selection_policy': 'Fail closed: no final selection until independent ph

In [29]:
from mostgen.reporting import build_reports
from mostgen.validation import verify_experiment
from mostgen.provenance import write_artifact_manifest

reports = build_reports(OUTPUT_DIR, config)
verification = verify_experiment(OUTPUT_DIR, config)
artifact_manifest = write_artifact_manifest(OUTPUT_DIR)
display({
    "acceptance_passed": verification["passed"],
    "checks": verification["checks"],
    "artifact_count": len(artifact_manifest["artifacts"]),
    "reports": reports,
})


{'acceptance_passed': True,
 'checks': {'all_expected_runs_present': True,
  'minimum_unique_per_run': True,
  'exact_matched_reviewer_budget': True,
  'all_three_families': True,
  'zero_psoralen_cores': True,
  'zero_known_phototoxic_matches': True,
  'no_uncertain_phototoxicity_selected': True,
  'no_iso_claims': True,
  'required_generated_schema': True,
  'reward_evaluator_separation': True,
  'metrics_present': True,
  'report_present': True,
  'presentation_present': True,
  'gpu_budget_respected': True},
 'artifact_count': 66,
 'reports': {'report': '/home/sigmatau17/TEST/runs/notebook-smoke/report/report.md',
  'presentation': '/home/sigmatau17/TEST/runs/notebook-smoke/report/presentation_7min.md'}}

In [30]:
from collections import Counter

generated = read_csv(generated_path)
run_counts = Counter((row["method_id"], row["seed"]) for row in generated)
unique_counts = {
    key: len({row["smiles"] for row in generated if (row["method_id"], row["seed"]) == key})
    for key in run_counts
}
failure_counts = Counter()
for row in generated:
    for reason in filter(None, row.get("failure_reasons", "").split(";")):
        failure_counts[reason] += 1

diagnostic = {
    "rows": len(generated),
    "exact_run_counts": dict(run_counts),
    "unique_per_run": unique_counts,
    "families": dict(Counter(row["family"] for row in generated)),
    "psoralen_alerts": sum(row["psoralen_alert"].lower() == "true" for row in generated),
    "known_phototoxic_matches": sum(row["known_phototoxic_match"].lower() == "true" for row in generated),
    "nonzero_rewards": sum(float(row["reward"]) > 0 for row in generated),
    "joint_pass_reward_models": sum(row["joint_pass"].lower() == "true" for row in generated),
    "top_failure_reasons": failure_counts.most_common(),
    "independent_provisional": review_result["independent_joint_pass"],
    "final_selected": review_result["selected_after_physical_oracle"],
}
display(diagnostic)
assert verification["passed"]


{'rows': 120,
 'exact_run_counts': {('prior_random', '17'): 40,
  ('weighted_retraining', '17'): 40,
  ('libinvent_rl', '17'): 40},
 'unique_per_run': {('prior_random', '17'): 40,
  ('weighted_retraining', '17'): 40,
  ('libinvent_rl', '17'): 40},
 'families': {'dewar_pyrimidinone': 42, 'nbd_qc': 39, 'spiropyran': 39},
 'psoralen_alerts': 0,
 'known_phototoxic_matches': 0,
 'nonzero_rewards': 120,
 'joint_pass_reward_models': 3,
 'top_failure_reasons': [('uv_joint_lcb_or_lambda', 98),
  ('phototoxicity_uncertain', 65),
  ('most_energy_or_half_life', 65),
  ('phototoxicity_risk', 2)],
 'independent_provisional': 1,
 'final_selected': 0}